# Thesis Replication Master Notebook

This notebook reproduces the full empirical pipeline of the thesis, including data preprocessing, market regime identification, portfolio optimization variants, adaptive portfolio construction, performance evaluation, and final reproducibility checks.
## Notebook Structure

1. Data Loading and Preprocessing  
   - Import Libraries  
   - Fixed Parameters for Thesis Replication  
   - Data Loading and Initial Preview  
   - Rolling Window and Out-of-Sample Evaluation Framework  

2. Market Environment Analysis  
   - Market Signals & Metrics  
   - Out-of-Sample Benchmark Returns  
   - Verification of Out-of-Sample Benchmark Returns  
   - PCA Transformation of Market Signals  
   - Cluster Sensitivity Analysis in PCA Space  
   - K-Means and GMM Market Regime Classification  
   - Regime Probabilities and Uncertainty Index  
   - OOS Benchmark Returns by Market Regime  
   - Descriptive Statistics of OOS Returns by Market Regime  
   - Statistical Significance Tests of OOS Returns across Regimes  
   - PCA-GMM Diagnostic Summary Figure  

3. Covariance Estimators - Covariance Matrices  
   - Sample VCV Matrix  
   - EWMA VCV Matrix  
   - Optional Robustness Check: Ledoit-Wolf Shrinkage Covariance Matrix  

4. Markowitz Mean-Variance Optimization Variants  
   - Markowitz Optimization Framework  
   - MARK_WF - Sample Covariance, No Risk-Free Rate, No Weight Cap  
   - MARK_WF_RF - Sample Covariance, Risk-Free Rate, No Weight Cap  
   - MARK15_S - Sample Covariance, 15% Weight Cap, No Risk-Free Rate  
   - MARK15_S_RF - Sample Covariance, 15% Weight Cap, Risk-Free Rate  
   - MARK15_EWMA - EWMA Covariance, 15% Weight Cap, No Risk-Free Rate  
   - MARK5_EWMA - EWMA Covariance, 5% Weight Cap, No Risk-Free Rate  
   - Markowitz Results Consolidation and Export  

5. Risk Parity Portfolio Optimization Variants  
   - Risk Parity Optimization Framework  
   - RP_S - Sample Covariance Risk Parity  
   - RP_EWMA - EWMA Covariance Risk Parity  
   - Risk Parity Results Consolidation and Export  

6. CVaR Portfolio Optimization Variants  
   - CVaR Optimization Framework  
   - CVAR 95-10 - 95% Confidence Level, 10% Weight Cap  
   - CVAR 99-15 - 99% Confidence Level, 15% Weight Cap  
   - CVaR Results Consolidation and Export  

7. Final Analysis Dataset and Baseline Evaluation  
   - Build Final ANALYSIS Dataset  
   - Capital Paths of Static Strategies and Benchmark  
   - Table 6 - Overall Performance Metrics  
   - Table 7 - Regime-Level Best Model Selection  
   - Export Final Analysis Workbook  

8. Adaptive Portfolio Simulation  
   - Adaptive Selection Rule  
   - Adaptive Portfolio Returns  
   - Adaptive Portfolio Capital Path versus Benchmark  
   - Export Adaptive Portfolio Results  

9. Adaptive Portfolio versus Static Strategies  
   - Consolidated Capital Paths  
   - Performance Comparison Including Adaptive Portfolio  
   - Ranking of Strategies  

10. Extended Performance Analysis  
    - Drawdown Analysis  
    - Recovery Ratio Analysis  
    - Risk-Return-Recovery Map  

11. Final Outputs and Reproducibility Checks  
    - Core Object Validation  
    - Reproducibility Summary  
    - Output File Inventory  

# 1. DATA LOADING AND PREPROCESSING

## Import Libraries

In [ ]:
import os
from itertools import combinations

import cvxpy as cp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import seaborn as sns

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.optimize import minimize
from scipy.stats import kruskal, mannwhitneyu

from sklearn.cluster import KMeans
from sklearn.covariance import LedoitWolf
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

## Fixed Parameters for Thesis Replication

In [ ]:
# 2. Fixed Parameters for Thesis Replication

# File paths
data_path = "FDATA.xlsx"
output_path = "output/"
os.makedirs(output_path, exist_ok=True)

# Fixed walk-forward parameters used in the thesis
lookback_period_months = 12
rebalancing_period_months = 3

# Fixed sample period used in the thesis
thesis_start_date = pd.Timestamp("2017-10-01")
thesis_end_date = pd.Timestamp("2024-03-28")

# Backward-compatible aliases
start_date = thesis_start_date
end_date = thesis_end_date

# Fixed quarterly rebalancing dates used in the thesis
rebalancing_dates = pd.date_range(
    start="2018-10-01",
    end=thesis_end_date,
    freq="QS"
)

## Data Loading and Initial Preview

In [ ]:
# 3. Data Loading and Initial Preview

df = pd.read_excel(data_path,sheet_name="RDATA",parse_dates=["DATE"])

df.set_index("DATE", inplace=True)

# Restrict dataset to the fixed thesis period
df = df.loc[start_date:end_date]

# Initial data check
display(df.head())
display(df.tail())

## Rolling Window and Out-of-Sample Evaluation Framework

In [ ]:
# 4. Rolling Window and Out-of-Sample Evaluation Framework

def get_lookback_window(df, cut_off_date, months=lookback_period_months):
    """Return the lookback window used for parameter estimation."""
    window_start = cut_off_date - pd.DateOffset(months=months)
    window_end = cut_off_date - pd.DateOffset(days=1)
    return df.loc[window_start:window_end]


def get_oos_window(df, cut_off_date, months=rebalancing_period_months):
    """Return the out-of-sample window used for portfolio evaluation."""
    window_start = cut_off_date
    window_end = cut_off_date + pd.DateOffset(months=months) - pd.DateOffset(days=1)
    return df.loc[window_start:window_end]


# Verification of rolling windows
for i, reb_date in enumerate(rebalancing_dates):
    lookback_df = get_lookback_window(df, reb_date)
    oos_df = get_oos_window(df, reb_date)

    print(f"Rolling window {i + 1}")
    print(f"Lookback: {lookback_df.index.min()} έως {lookback_df.index.max()}")
    print(f"Out-of-Sample: {oos_df.index.min()} έως {oos_df.index.max()}")

# 2. MARKET ENVIRONMENT ANALYSIS 

## Market Signals & Metrics

In [ ]:
# compute market signals (volatility, correlation, trend, skewness, kurtosis)
def compute_market_signals(returns_window, benchmark_returns):
    vol = benchmark_returns.std() * np.sqrt(252)  # annualized volatility
    corr_matrix = returns_window.corr()
    avg_corr = corr_matrix.iloc[1:, 1:].values[np.triu_indices_from(corr_matrix.iloc[1:, 1:], 1)].mean()  # Μέση συσχέτιση μεταξύ των μετοχών, εξαιρώντας τον δείκτη SX5E
    trend = (benchmark_returns + 1).prod() - 1  # cumulative return
    skew = benchmark_returns.skew()  #  (skewness)
    kurt = benchmark_returns.kurtosis()  #  (excess kurtosis by default)
    
    return vol, avg_corr, trend, skew, kurt

signals_list = []  # Create a list to store market signals

# Rolling loop displaying key signals and window periods
for i, reb_date in enumerate(rebalancing_dates):
    lookback_df = get_lookback_window(df, reb_date, months=lookback_period_months)
    oos_df = get_oos_window(df, reb_date, months=rebalancing_period_months)
    
    # compute log returns
    lookback_returns = np.log(lookback_df / lookback_df.shift(1)).dropna()
    oos_returns = np.log(oos_df / oos_df.shift(1)).dropna()
    
    # Benchmark returns from the first data column after the DATE/index (SX5E)
    benchmark_lookback = lookback_returns.iloc[:, 0] 
    benchmark_oos = oos_returns.iloc[:, 0] 
    
    # Market signals 
    vol, avg_corr, trend, skew, kurt = compute_market_signals(lookback_returns, benchmark_lookback)
    
  

    # Convert datetime values to dates for clean display
    lookback_start = lookback_df.index.min().date()
    lookback_end = lookback_df.index.max().date()
    oos_start = oos_df.index.min().date()
    oos_end = oos_df.index.max().date()

    # Display results
    print(f'Lookback: {lookback_start} έως {lookback_end}')
    print(f'Out-of-Sample: {oos_start} έως {oos_end}')
    print(f'Rolling window {i+1}')
    print(f'Market Volatility (annualized): {vol:.4f}')  # # Ετήσια μεταβλητότητα
    print(f'Average Cross-Correlation: {avg_corr:.4f}')  # # Μέση συσχέτιση
    print(f'Market Trend (cumulative return): {trend:.2%}')  # # Σωρευτική απόδοση
    print(f'Skewness (ασυμμετρία): {skew:.4f}')  # # Ασυμμετρία
    print(f'Kurtosis (κύρτωση): {kurt:.4f}')  # # Κύρτωση
    print('---')
    
    # Store signals with dates formatted as YYYY-MM-DD strings without time
    signals_list.append({
        'Rolling Window': i+1,
        'Lookback Start': lookback_start.strftime('%Y-%m-%d'),
        'Lookback End': lookback_end.strftime('%Y-%m-%d'),
        'Out-of-Sample Start': oos_start.strftime('%Y-%m-%d'),
        'Out-of-Sample End': oos_end.strftime('%Y-%m-%d'),
        'Market Volatility': vol,
        'Average Cross-Correlation': avg_corr,
        'Market Trend': trend,
        'Skewness': skew,
        'Kurtosis': kurt
    })

# Convert list to a Pandas DataFrame
signals_df = pd.DataFrame(signals_list)

# Export results to Excel
export_signals = True

if export_signals:
   signals_df.to_excel(output_path + "SIGNALS.xlsx",index=False)

display(signals_df)

## Out-of-Sample Benchmark Returns

In [ ]:
# Out-of-Sample Benchmark Returns

# Create a list to store benchmark net returns for the out-of-sample periods
oos_returns_list = []

for i, reb_date in enumerate(rebalancing_dates):
    oos_df = get_oos_window(df,reb_date,months=rebalancing_period_months)
    
    # Benchmark index price (SX5E) at the beginning and end of the OOS period
    start_price = oos_df.iloc[0, 0]
    end_price = oos_df.iloc[-1, 0]
    
    # Compute net return for the OOS period
    oos_net_return = (end_price / start_price) - 1
    
    oos_returns_list.append({
        "Rolling Window": i + 1,
        "Out-of-Sample Start": oos_df.index.min().strftime("%Y-%m-%d"),
        "Out-of-Sample End": oos_df.index.max().strftime("%Y-%m-%d"),
        "OOS Net Return": oos_net_return
    })

# Convert list to DataFrame
oos_returns_df = pd.DataFrame(oos_returns_list)

# Display results
display(oos_returns_df.head())

# Export results to Excel
export_oos_returns = True

if export_oos_returns:
    oos_returns_df.to_excel(output_path + "OOS_Net_Returns.xlsx",index=False)

## Verification of Out-of-Sample Benchmark Returns

In [ ]:
for i, reb_date in enumerate(rebalancing_dates):
    oos_df = get_oos_window(df, reb_date, months=rebalancing_period_months)
    start_date = oos_df.index.min().date()
    end_date = oos_df.index.max().date()
    start_price = oos_df.iloc[0, 0]
    end_price = oos_df.iloc[-1, 0]
    net_return = (end_price / start_price) - 1
    
    display(f"Window {i+1}: From {start_date} (Price: {start_price}) to {end_date} (Price: {end_price}) => Return: {net_return*100:.2f}%")


## PCA Transformation of Market Signals

In [ ]:
# 1. Load market signals
signals_df = pd.read_excel(output_path + "SIGNALS.xlsx")

# 2. Select features
features = signals_df[['Market Volatility', 'Average Cross-Correlation','Market Trend', 'Skewness', 'Kurtosis']]

# 3. Standardization
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# 4. Initial PCA with 3 components for explained variance and loading analysis
pca = PCA(n_components=3)
pca_features_3 = pca.fit_transform(features_scaled)
explained_var = np.cumsum(pca.explained_variance_ratio_)

print("Explained variance ratio by component:", pca.explained_variance_ratio_)
print("Cumulative explained variance:", explained_var)

# PCA loading analysis
loadings = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2', 'PC3'],
    index=features.columns
)

print("\n--- PCA Loadings Table ---")
print("Shows the relationship between each variable and each principal component:")
print(loadings.round(4))

plt.figure(figsize=(8, 6))
sns.heatmap(loadings, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Variable Contributions to Principal Components')
plt.ylabel('Features')
plt.xlabel('Principal Components')
plt.tight_layout()
plt.show()

# 5. Final PCA with 2 components for clustering / visualization
pca_final = PCA(n_components=2)
pca_features = pca_final.fit_transform(features_scaled)

explained = np.cumsum(pca_final.explained_variance_ratio_)
signals_df = pd.read_excel(output_path + "SIGNALS.xlsx")
features = signals_df[
    [
        'Market Volatility',
        'Average Cross-Correlation',
        'Market Trend',
        'Skewness',
        'Kurtosis'
    ]
]


# PCA
pca = PCA(n_components=2)
pca_features = pca.fit_transform(features_scaled)

explained = np.cumsum(pca.explained_variance_ratio_)

print("Cumulative explained variance:")
print(explained)

# PCA cumulative explained variance table
pca_table = pd.DataFrame({
    "Principal Component": ["PC1", "PC2"],
    "Cumulative Explained Variance (%)": (explained * 100).round(2)
})

print("\nPCA Explained Variance Table:")
print(pca_table.to_string(index=False))

# Visualize market signals in PCA space
plt.figure(figsize=(8,6))
plt.scatter(pca_features[:,0],pca_features[:,1],alpha=0.7)
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("PCA Representation of Market Signals")
plt.show()

## Cluster Sensitivity Analysis in PCA Space

In [ ]:
# =========================
# Cluster Sensitivity Analysis in PCA Space
# =========================

# Compute Silhouette Scores for different k values
silhouette_scores = {}
K_range = range(2, 8)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42)
    labels_temp = kmeans_temp.fit_predict(pca_features)

    sil = silhouette_score(pca_features, labels_temp)
    silhouette_scores[k] = sil

    print(f"k={k}, Silhouette Score={sil:.3f}")

# Plot Silhouette Scores
plt.figure(figsize=(7, 4))

plt.plot(
    list(silhouette_scores.keys()),
    list(silhouette_scores.values()),
    marker="o"
)

plt.title("Silhouette Scores for Different Numbers of Clusters")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


# Elbow method for selecting the number of clusters based on WCSS
wcss = []
K_range = range(2, 10)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42)
    kmeans_temp.fit(pca_features)

    wcss.append(kmeans_temp.inertia_)

# Print WCSS values
print("\nElbow method (WCSS by k):")
for k, val in zip(K_range, wcss):
    print(f"k={k}, WCSS={val:.3f}")

# Elbow plot
plt.figure(figsize=(7, 4))

plt.plot(K_range, wcss, marker="o")

plt.title("Elbow Method: Within-Cluster Sum of Squares (WCSS)")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## K-Means and GMM Market Regime Classification

In [ ]:
# K-Means with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42)
kmeans_labels = kmeans.fit_predict(pca_features)

sil_kmeans = silhouette_score(pca_features, kmeans_labels)
print(f"Silhouette Score K-Means: {sil_kmeans:.3f}")

# Gaussian Mixture Model with 4 components
gmm = GaussianMixture(n_components=4, covariance_type="full", random_state=42)
gmm_labels = gmm.fit_predict(pca_features)

gmm_probs = gmm.predict_proba(pca_features)
sil_gmm = silhouette_score(pca_features, gmm_labels)

print(f"Silhouette Score GMM: {sil_gmm:.3f}")

# Compare K-Means and GMM in PCA space
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(
    pca_features[:, 0],
    pca_features[:, 1],
    c=kmeans_labels,
    cmap="tab10",
    edgecolor="k",
    alpha=0.7
)

axes[0].set_title(f"K-Means (k=4) | Silhouette={sil_kmeans:.3f}")
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

axes[1].scatter(
    pca_features[:, 0],
    pca_features[:, 1],
    c=gmm_labels,
    cmap="tab10",
    edgecolor="k",
    alpha=0.7
)

axes[1].set_title(f"Gaussian Mixture (4 regimes) | Silhouette={sil_gmm:.3f}")
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")

plt.tight_layout()
plt.show()

## Regime Probabilities and Uncertainty Index 

In [ ]:
# Probability heatmap (GMM confidence)
probs_df = pd.DataFrame(gmm_probs, columns=[f"Regime {i}" for i in range(4)])

plt.figure(figsize=(10, 6))
sns.heatmap(
    probs_df.T,
    cmap="viridis",
    cbar_kws={"label": "Probability"}
)

plt.xticks(
    ticks=np.arange(len(probs_df)) + 0.5,
    labels=np.arange(1, len(probs_df) + 1)
)

plt.title("Market Regime Probabilities by Rolling Window")
plt.xlabel("Time Windows (sequential order)")
plt.ylabel("Regimes")
plt.tight_layout()
plt.show()


# Enriched DataFrame with labels and probabilities
signals_df["Regime_GMM"] = gmm_labels

for i in range(4):
    signals_df[f"Prob_Regime_{i}"] = gmm_probs[:, i]


# Ambiguity / uncertainty index
signals_df["Uncertainty_Index"] = 1 - gmm_probs.max(axis=1)

signals_df = signals_df.reset_index(drop=True)
signals_df["Window"] = np.arange(1, len(signals_df) + 1)


# Display the 10 most uncertain classification periods
top_uncertain = signals_df.nlargest(10, "Uncertainty_Index")[
    ["Rolling Window", "Lookback Start", "Lookback End", "Regime_GMM", "Uncertainty_Index"]
]

print("\n10 most uncertain classification periods:")
print(top_uncertain)


# Visualize uncertainty over time
plt.figure(figsize=(10, 4))

plt.plot(
    range(1, len(signals_df) + 1),
    signals_df["Uncertainty_Index"],
    color="red"
)

plt.title("Market Regime Uncertainty Index (1 - max(probability))")
plt.xlabel("Time Windows (sequential order)")
plt.ylabel("Uncertainty Index")
plt.tight_layout()
plt.show()

export_gmm_probabilities = True

if export_gmm_probabilities:
    signals_df.to_excel(
        output_path + "Market_Regimes_GMM_with_Probabilities.xlsx",
        index=False)

## OOS Benchmark Returns by Market Regime

In [ ]:
# Compute benchmark net out-of-sample returns by rolling window
oos_returns_list = []

for i, reb_date in enumerate(rebalancing_dates):
    oos_df = get_oos_window(df, reb_date, months=rebalancing_period_months)

    start_price = oos_df.iloc[0, 0]
    end_price = oos_df.iloc[-1, 0]
    oos_net_return = (end_price / start_price) - 1

    oos_returns_list.append(
        {
            "Rolling Window": i + 1,
            "Out-of-Sample Start": oos_df.index.min().strftime("%Y-%m-%d"),
            "Out-of-Sample End": oos_df.index.max().strftime("%Y-%m-%d"),
            "OOS Net Return": oos_net_return
        }
    )

oos_returns_df = pd.DataFrame(oos_returns_list)

# Merge OOS returns with final GMM regimes
merged_df = pd.merge(
    oos_returns_df,
    signals_df[
        [
            "Rolling Window",
            "Regime_GMM",
            "Uncertainty_Index"
        ]
    ],
    on="Rolling Window",
    how="left"
)

print(
    merged_df[
        ["Rolling Window", "OOS Net Return", "Regime_GMM", "Uncertainty_Index"]
    ].to_string(
        index=False,
        formatters={
            "OOS Net Return": "{:.6f}".format,
            "Uncertainty_Index": "{:.6f}".format
        }
    )
)

export_oos_regimes = True

if export_oos_regimes:
    merged_df.to_excel(
        output_path + "OOS_Net_Returns_with_Regimes.xlsx",
        index=False
    )

## Descriptive Statistics of OOS Returns by Market Regime

In [ ]:
# Compute mean return by GMM Market Regime
mean_returns_per_regime = merged_df.groupby("Regime_GMM")["OOS Net Return"].mean()

print("Mean return by regime:")
print(mean_returns_per_regime.to_string(float_format="{:.6f}".format))

# Bar plot of mean return by regime
plt.figure(figsize=(8, 5))
mean_returns_per_regime.plot(kind="bar", color="skyblue")
plt.title("Mean Out-of-Sample Return by Market Regime")
plt.xlabel("Market Regime")
plt.ylabel("Mean Out-of-Sample Return")
plt.show()

# Boxplot of returns by regime
plt.figure(figsize=(10, 6))
sns.boxplot(
    x="Regime_GMM",
    y="OOS Net Return",
    hue="Regime_GMM",
    data=merged_df,
    palette="Set2",
    legend=False
)
plt.title("Distribution of Out-of-Sample Returns by Market Regime")
plt.xlabel("Market Regime")
plt.ylabel("Out-of-Sample Return")
plt.show()

# Table with mean values
mean_returns_table = (
    mean_returns_per_regime
    .reset_index()
    .rename(columns={"OOS Net Return": "Mean OOS Return"})
)

print("Mean return table by regime:")
print(mean_returns_table.to_string(index=False, float_format="{:.6f}".format))

# Basic return statistics by regime
stats_table = merged_df.groupby("Regime_GMM")["OOS Net Return"].describe()

print("\nBasic return statistics by regime:")
print(stats_table.to_string(float_format="{:.6f}".format))

## Statistical Significance Tests of OOS Returns across Regimes

In [ ]:
# Statistical significance test of OOS Returns by GMM Market Regime

# Check that the required columns exist
required_columns = {"OOS Net Return", "Regime_GMM"}

if not required_columns.issubset(merged_df.columns):
    raise ValueError(
        "merged_df must contain the columns "
        "'OOS Net Return' and 'Regime_GMM'."
    )

# Clean data
df_stats = merged_df[["OOS Net Return", "Regime_GMM"]].dropna()

# Kruskal–Wallis test
groups = [
    group["OOS Net Return"].values
    for _, group in df_stats.groupby("Regime_GMM")
]

H, p_kw = kruskal(*groups)

print(f"\nKruskal–Wallis: H={H:.3f}, p-value={p_kw:.4f}")

# Post-hoc Mann–Whitney U with Holm–Bonferroni correction,
# only if the overall test is statistically significant
if p_kw < 0.05 and df_stats["Regime_GMM"].nunique() > 2:
    pairs = list(combinations(sorted(df_stats["Regime_GMM"].unique()), 2))
    results = []

    for a, b in pairs:
        xa = df_stats.loc[df_stats["Regime_GMM"] == a, "OOS Net Return"].values
        xb = df_stats.loc[df_stats["Regime_GMM"] == b, "OOS Net Return"].values

        U, p = mannwhitneyu(xa, xb, alternative="two-sided")

        results.append(
            {
                "pair": f"{a} vs {b}",
                "U": U,
                "p_raw": p
            }
        )

    df_post = (
        pd.DataFrame(results)
        .sort_values("p_raw")
        .reset_index(drop=True)
    )

    m = len(df_post)

    df_post["alpha_i"] = 0.05 / (m - df_post.index)
    df_post["p_adj_holm"] = np.maximum.accumulate(
        df_post["p_raw"] * (m - df_post.index)
    )
    df_post["significant"] = df_post["p_raw"] <= df_post["alpha_i"]

    print("\nPost-hoc Mann–Whitney tests with Holm–Bonferroni correction:")
    print(
        df_post[
            ["pair", "U", "p_raw", "p_adj_holm", "alpha_i", "significant"]
        ].to_string(index=False, float_format="{:.6f}".format)
    )

else:
    print("Post-hoc tests are not required or the overall test is not statistically significant.")

## PCA-GMM Diagnostic Summary Figure

In [ ]:
# Create a summary PCA-GMM figure with 3 panels
fig = plt.figure(figsize=(16, 12), constrained_layout=True)

gs = gridspec.GridSpec(
    3,
    1,
    figure=fig,
    height_ratios=[1, 1, 0.5]
)

# 1. PCA scatter plot with GMM regime labels
ax0 = plt.subplot(gs[0])

colors = ["tab:blue", "tab:orange", "tab:green", "tab:red"]

for regime, color in zip(sorted(np.unique(gmm_labels)), colors):
    ax0.scatter(
        pca_features[gmm_labels == regime, 0],
        pca_features[gmm_labels == regime, 1],
        color=color,
        label=f"Regime {regime}",
        edgecolor="k",
        alpha=0.7
    )

ax0.set_xlabel("PC1")
ax0.set_ylabel("PC2")
ax0.set_title("Market Regimes in PCA Space with GMM Classification")
ax0.legend(title="Market Regime")
ax0.grid(True, linestyle="--", alpha=0.4)


# 2. GMM probability heatmap
ax1 = plt.subplot(gs[1])

sns.heatmap(
    probs_df.T,
    cmap="viridis",
    cbar_kws={"label": "Probability"},
    ax=ax1
)

ax1.set_xlabel("Rolling Window")
ax1.set_ylabel("Regimes")
ax1.set_title("Market Regime Probabilities by Rolling Window")
ax1.set_xticks(np.arange(len(probs_df)) + 0.5)
ax1.set_xticklabels(np.arange(1, len(probs_df) + 1))


# 3. Uncertainty Index
ax2 = plt.subplot(gs[2])

ax2.plot(
    range(1, len(gmm_probs) + 1),
    1 - gmm_probs.max(axis=1),
    color="red"
)

ax2.set_xlabel("Rolling Window")
ax2.set_ylabel("Uncertainty Index")
ax2.set_title("Market Regime Uncertainty Index (1 - max(probability))")
ax2.grid(True, linestyle="--", alpha=0.4)

plt.show()

# 3. COVARIANCE ESTIMATORS - COVARIANCE MATRICES

## Sample VCV Matrix

In [ ]:
# =========================
# Sample Variance-Covariance Matrices
# =========================

# Use the price DataFrame already loaded in the initial data section.
# If the notebook is run from this point only, reload the data from the Excel file.
try:
    df_prices_full = df.copy()
except NameError:
    df_prices_full = pd.read_excel(
        data_path,
        sheet_name="RDATA",
        parse_dates=["DATE"]
    )
    df_prices_full.set_index("DATE", inplace=True)

# Select stock price columns.
# The first column is the benchmark index, so it is excluded from the covariance matrices.
stock_columns = df_prices_full.columns[1:]
stocks_df = df_prices_full.loc[:, stock_columns]

# Build rolling lookback windows.
# Convert date boundaries to pandas Timestamp objects for consistent comparisons.
comparison_start_date = thesis_start_date
comparison_end_date = thesis_end_date

rolling_windows = []

for rebalancing_date in rebalancing_dates:
    rebalancing_date = pd.Timestamp(rebalancing_date)

    window_start = rebalancing_date - pd.DateOffset(months=lookback_period_months)
    window_end = rebalancing_date - pd.DateOffset(days=1)

    if window_start >= comparison_start_date and window_end <= comparison_end_date:
        rolling_windows.append((window_start, window_end))
        

print(f"Number of rolling windows created: {len(rolling_windows)}")

# Estimate sample variance-covariance matrices for each rolling window.
sample_cov_matrices = []

for window_number, (window_start, window_end) in enumerate(rolling_windows, start=1):
    window_prices = stocks_df.loc[window_start:window_end]

    log_returns = np.log(window_prices / window_prices.shift(1)).dropna()

    sample_cov_matrix = log_returns.cov()

    sample_cov_matrices.append(
        {
            "Rolling Window": window_number,
            "Start Date": window_start.strftime("%Y-%m-%d"),
            "End Date": window_end.strftime("%Y-%m-%d"),
            "Covariance Matrix": sample_cov_matrix,
            "Dimensions": sample_cov_matrix.shape,
        }
    )

# Display the first sample covariance matrix as a diagnostic check.
first_cov_matrix = sample_cov_matrices[0]["Covariance Matrix"]

print(
    "Sample covariance matrix for Rolling Window 1 "
    f"({sample_cov_matrices[0]['Start Date']} to {sample_cov_matrices[0]['End Date']}):"
)
print(first_cov_matrix)
print(f"Dimensions: {sample_cov_matrices[0]['Dimensions']}")

# Optional export to Excel.
export_sample_cov_matrices = True

if export_sample_cov_matrices:
    export_file = output_path + "VCV_SAMPLE.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        for item in sample_cov_matrices:
            sheet_name = f"Window_{item['Rolling Window']}"
            item["Covariance Matrix"].to_excel(writer, sheet_name=sheet_name)

    print(f"Sample covariance matrices exported to {export_file}")

## EWMA VCV Matrix

In [ ]:
# =========================
# EWMA Variance-Covariance Matrices
# =========================

# EWMA decay factor.
# The value 0.94 is commonly used in RiskMetrics-type volatility modelling.
ewma_lambda = 0.94


def compute_ewma_covariance_matrix(returns, decay_factor):
    """
    Estimate an EWMA covariance matrix from a return matrix.

    Parameters
    ----------
    returns : pandas.DataFrame
        Asset return matrix, with dates as rows and assets as columns.

    decay_factor : float
        EWMA decay factor. Higher values give more weight to older observations.

    Returns
    -------
    pandas.DataFrame
        EWMA covariance matrix.
    """
    n_assets = returns.shape[1]

    # Initialize the covariance matrix using the first return observation.
    first_return = returns.iloc[0].values.reshape(n_assets, 1)
    ewma_covariance = first_return @ first_return.T

    # Recursively update the covariance matrix.
    for t in range(1, len(returns)):
        current_return = returns.iloc[t].values.reshape(n_assets, 1)

        ewma_covariance = (
            decay_factor * ewma_covariance
            + (1 - decay_factor) * (current_return @ current_return.T)
        )

    return pd.DataFrame(
        ewma_covariance,
        index=returns.columns,
        columns=returns.columns,
    )


# Estimate EWMA variance-covariance matrices for each rolling window.
ewma_cov_matrices = []

for window_number, (window_start, window_end) in enumerate(rolling_windows, start=1):
    window_prices = stocks_df.loc[window_start:window_end]

    log_returns = np.log(window_prices / window_prices.shift(1)).dropna()

    ewma_cov_matrix = compute_ewma_covariance_matrix(
        returns=log_returns,
        decay_factor=ewma_lambda,
    )

    ewma_cov_matrices.append(
        {
            "Rolling Window": window_number,
            "Start Date": window_start.strftime("%Y-%m-%d"),
            "End Date": window_end.strftime("%Y-%m-%d"),
            "EWMA Covariance Matrix": ewma_cov_matrix,
            "Dimensions": ewma_cov_matrix.shape,
        }
    )

# Display the first EWMA covariance matrix as a diagnostic check.
first_ewma_cov_matrix = ewma_cov_matrices[0]["EWMA Covariance Matrix"]

print(
    "EWMA covariance matrix for Rolling Window 1 "
    f"({ewma_cov_matrices[0]['Start Date']} to {ewma_cov_matrices[0]['End Date']}):"
)
print(first_ewma_cov_matrix)
print(f"Dimensions: {ewma_cov_matrices[0]['Dimensions']}")

# Optional export to Excel.
export_ewma_cov_matrices = True

if export_ewma_cov_matrices:
    export_file = output_path + "VCV_EWMA.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        for item in ewma_cov_matrices:
            sheet_name = f"Window_{item['Rolling Window']}"
            item["EWMA Covariance Matrix"].to_excel(writer, sheet_name=sheet_name)

    print(f"EWMA covariance matrices exported to {export_file}")

## Optional Robustness Check: Ledoit-Wolf Shrinkage Covariance Matrix

In [ ]:
# =========================
# Optional Robustness Check: Ledoit-Wolf Shrinkage Covariance Matrices
# =========================

# This estimator is computed as an optional robustness check.
# It is not used in the core portfolio construction pipeline unless explicitly enabled.

compute_ledoit_wolf_cov_matrices = False

if compute_ledoit_wolf_cov_matrices:
    ledoit_wolf_model = LedoitWolf()

    ledoit_wolf_cov_matrices = []

    for window_number, (window_start, window_end) in enumerate(rolling_windows, start=1):
        window_prices = stocks_df.loc[window_start:window_end]

        log_returns = np.log(window_prices / window_prices.shift(1)).dropna()

        ledoit_wolf_model.fit(log_returns)

        ledoit_wolf_cov_matrix = pd.DataFrame(
            ledoit_wolf_model.covariance_,
            index=log_returns.columns,
            columns=log_returns.columns,
        )

        ledoit_wolf_cov_matrices.append(
            {
                "Rolling Window": window_number,
                "Start Date": window_start.strftime("%Y-%m-%d"),
                "End Date": window_end.strftime("%Y-%m-%d"),
                "Ledoit-Wolf Covariance Matrix": ledoit_wolf_cov_matrix,
                "Dimensions": ledoit_wolf_cov_matrix.shape,
            }
        )

    # Display the first Ledoit-Wolf covariance matrix as a diagnostic check.
    first_ledoit_wolf_cov_matrix = ledoit_wolf_cov_matrices[0][
        "Ledoit-Wolf Covariance Matrix"
    ]

    print(
        "Ledoit-Wolf covariance matrix for Rolling Window 1 "
        f"({ledoit_wolf_cov_matrices[0]['Start Date']} to "
        f"{ledoit_wolf_cov_matrices[0]['End Date']}):"
    )
    print(first_ledoit_wolf_cov_matrix)
    print(f"Dimensions: {ledoit_wolf_cov_matrices[0]['Dimensions']}")

    # Optional export to Excel.
    export_ledoit_wolf_cov_matrices = False

    if export_ledoit_wolf_cov_matrices:
        export_file = output_path + "VCV_LEDOIT_WOLF.xlsx"

        with pd.ExcelWriter(export_file) as writer:
            for item in ledoit_wolf_cov_matrices:
                sheet_name = f"Window_{item['Rolling Window']}"
                item["Ledoit-Wolf Covariance Matrix"].to_excel(
                    writer,
                    sheet_name=sheet_name,
                )

        print(f"Ledoit-Wolf covariance matrices exported to {export_file}")

# 4. MARKOWITZ Mean-Variance Optimization Variants

## Markowitz Optimization Framework

In [ ]:
# =========================
# Markowitz Optimization Framework
# =========================

# Risk-free rates by rolling window, as used in the thesis.
# Each value corresponds to one rolling window.
risk_free_rates_by_window = [
    0.0047, 0.00246, -0.00068, -0.00328, -0.00572, -0.00187,
    -0.00469, -0.00453, -0.00521, -0.00575, -0.00292, -0.00203,
    -0.00191, -0.00179, 0.00547, 0.01368, 0.02109, 0.02645,
    0.02301, 0.02391, 0.02838, 0.02028
]

if len(risk_free_rates_by_window) != len(rolling_windows):
    raise ValueError(
        "The number of risk-free rates must match the number of rolling windows."
    )


def compute_log_returns(price_data):
    """
    Compute log returns from a price DataFrame.
    """
    return np.log(price_data / price_data.shift(1)).dropna()


def compute_simple_returns(price_data):
    """
    Compute simple percentage returns from a price DataFrame.
    """
    return price_data.pct_change().dropna()


def negative_sharpe_ratio(weights, mean_returns, covariance_matrix, risk_free_rate):
    """
    Objective function for Markowitz optimization.

    The optimizer minimizes this function, so the Sharpe ratio is multiplied by -1.
    """
    portfolio_return = np.dot(mean_returns, weights)
    portfolio_volatility = np.sqrt(
        np.dot(weights.T, np.dot(covariance_matrix, weights))
    )

    if portfolio_volatility == 0:
        return 0

    return -(portfolio_return - risk_free_rate) / portfolio_volatility


def compute_oos_metrics(portfolio_returns, risk_free_rate=0.0, use_risk_free=False):
    """
    Compute out-of-sample portfolio performance metrics.
    """
    if portfolio_returns.empty:
        return {
            "cumulative_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe_ratio": np.nan,
            "max_drawdown": np.nan,
            "hit_ratio": np.nan,
        }

    cumulative_return = (1 + portfolio_returns).prod() - 1

    # Quarterly rebalancing: four out-of-sample periods per year.
    annual_return = (1 + cumulative_return) ** 4 - 1

    # Daily out-of-sample returns are annualized with sqrt(252).
    annual_volatility = portfolio_returns.std() * np.sqrt(252)

    if annual_volatility != 0:
        if use_risk_free:
            sharpe_ratio = (annual_return - risk_free_rate) / annual_volatility
        else:
            sharpe_ratio = annual_return / annual_volatility
    else:
        sharpe_ratio = np.nan

    capital_path = (1 + portfolio_returns).cumprod() * 100
    peak = capital_path.cummax()
    drawdown = (capital_path - peak) / peak
    max_drawdown = drawdown.min()

    hit_ratio = (portfolio_returns > 0).mean()

    return {
        "cumulative_return": cumulative_return,
        "annual_return": annual_return,
        "annual_volatility": annual_volatility,
        "sharpe_ratio": sharpe_ratio,
        "max_drawdown": max_drawdown,
        "hit_ratio": hit_ratio,
    }


def run_markowitz_variant(
    variant_name,
    covariance_matrices,
    covariance_matrix_key,
    max_weight=1.0,
    use_risk_free=False,
):
    """
    Run a Markowitz mean-variance optimization variant across all rolling windows.

    Parameters
    ----------
    variant_name : str
        Name of the Markowitz variant.

    covariance_matrices : list
        List of covariance matrix dictionaries produced in the covariance estimator section.

    covariance_matrix_key : str
        Dictionary key used to retrieve the covariance matrix.

    max_weight : float
        Upper bound for each asset weight.

    use_risk_free : bool
        If True, the risk-free rate is included in the objective function and Sharpe ratio.

    Returns
    -------
    dict
        Dictionary containing optimized weights, window-level metrics, and capital path.
    """
    optimized_weights = []
    window_metrics = []
    capital_series = []

    current_capital = 100

    for window_index, ((window_start, window_end), covariance_item) in enumerate(
        zip(rolling_windows, covariance_matrices),
        start=1,
    ):
        risk_free_rate = (
            risk_free_rates_by_window[window_index - 1]
            if use_risk_free
            else 0.0
        )

        covariance_matrix = covariance_item[covariance_matrix_key]
        covariance_matrix = covariance_matrix.loc[stock_columns, stock_columns]

        window_prices = stocks_df.loc[window_start:window_end]
        log_returns = compute_log_returns(window_prices)
        mean_returns = log_returns.mean().values

        initial_weights = np.full(len(stock_columns), 1 / len(stock_columns))

        constraints = {
            "type": "eq",
            "fun": lambda weights: np.sum(weights) - 1,
        }

        bounds = [(0, max_weight) for _ in stock_columns]

        optimization_result = minimize(
            negative_sharpe_ratio,
            initial_weights,
            args=(mean_returns, covariance_matrix.values, risk_free_rate),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
        )

        if optimization_result.success:
            weights = optimization_result.x
        else:
            weights = np.full(len(stock_columns), np.nan)
            print(
                f"Optimization failed for {variant_name}, "
                f"Rolling Window {window_index}, ending {window_end.strftime('%Y-%m-%d')}."
            )

        optimized_weights.append(weights)

        oos_start = window_end + pd.Timedelta(days=1)
        oos_end = oos_start + pd.DateOffset(months=rebalancing_period_months) - pd.Timedelta(days=1)

        if oos_end > stocks_df.index[-1]:
            oos_end = stocks_df.index[-1]

        oos_prices = stocks_df.loc[oos_start:oos_end]
        oos_returns = compute_simple_returns(oos_prices)

        if oos_returns.empty or np.isnan(weights).all():
            portfolio_returns = pd.Series(dtype=float)
        else:
            portfolio_returns = oos_returns.dot(weights)

        metrics = compute_oos_metrics(
            portfolio_returns,
            risk_free_rate=risk_free_rate,
            use_risk_free=use_risk_free,
        )

        window_metrics.append(
            {
                "Variant": variant_name,
                "Rolling Window": window_index,
                "Lookback Start": window_start.strftime("%Y-%m-%d"),
                "Lookback End": window_end.strftime("%Y-%m-%d"),
                "OOS Start": oos_prices.index.min().strftime("%Y-%m-%d")
                if not oos_prices.empty
                else None,
                "OOS End": oos_prices.index.max().strftime("%Y-%m-%d")
                if not oos_prices.empty
                else None,
                "Risk-Free Rate": risk_free_rate,
                "Optimization Success": optimization_result.success,
                "cumulative_return": metrics["cumulative_return"],
                "annual_return": metrics["annual_return"],
                "annual_volatility": metrics["annual_volatility"],
                "sharpe_ratio": metrics["sharpe_ratio"],
                "max_drawdown": metrics["max_drawdown"],
                "hit_ratio": metrics["hit_ratio"],
            }
        )

        if portfolio_returns.empty:
            for date in oos_prices.index:
                capital_series.append(
                    {
                        "Date": date,
                        "Variant": variant_name,
                        "Portfolio Value": current_capital,
                    }
                )
        else:
            for date, portfolio_return in portfolio_returns.items():
                current_capital *= 1 + portfolio_return

                capital_series.append(
                    {
                        "Date": date,
                        "Variant": variant_name,
                        "Portfolio Value": current_capital,
                    }
                )

    weights_df = pd.DataFrame(
        optimized_weights,
        columns=stock_columns,
    )
    weights_df.insert(0, "Rolling Window", range(1, len(weights_df) + 1))
    weights_df.insert(0, "Variant", variant_name)

    window_metrics_df = pd.DataFrame(window_metrics)

    capital_path_df = pd.DataFrame(capital_series)

    return {
        "variant_name": variant_name,
        "weights": weights_df,
        "window_metrics": window_metrics_df,
        "capital_path": capital_path_df,
    }
def plot_markowitz_metric_distributions(window_metrics_df, variant_name):
    """
    Plot the distribution of key rolling-window performance metrics
    in a single consolidated figure.
    """
    metric_specs = [
        ("cumulative_return", "Cumulative Return"),
        ("annual_return", "Annualized Return"),
        ("annual_volatility", "Annualized Volatility"),
        ("sharpe_ratio", "Sharpe Ratio"),
        ("max_drawdown", "Max Drawdown"),
        ("hit_ratio", "Hit Ratio"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for ax, (metric_column, metric_label) in zip(axes, metric_specs):
        ax.hist(
            window_metrics_df[metric_column].dropna(),
            bins=30,
            alpha=0.7,
            edgecolor="black"
        )
        ax.set_title(metric_label)
        ax.set_xlabel(metric_label)
        ax.set_ylabel("Frequency")
        ax.grid(True)

    fig.suptitle(
        f"Distribution of Rolling-Window Performance Metrics - {variant_name}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()

## MARK_WF - Sample Covariance, No Risk-Free Rate, No Weight Cap

In [ ]:
# =========================
# MARK_WF - Sample Covariance, No Risk-Free Rate, No Weight Cap
# =========================

variant_name = "MARK_WF"

# Create the main Markowitz results dictionary if it does not already exist.
try:
    markowitz_results
except NameError:
    markowitz_results = {}

# Run the Markowitz variant.
markowitz_results[variant_name] = run_markowitz_variant(
    variant_name=variant_name,
    covariance_matrices=sample_cov_matrices,
    covariance_matrix_key="Covariance Matrix",
    max_weight=1.0,
    use_risk_free=False,
)

# Extract results for easier use.
mark_wf_results = markowitz_results[variant_name]
mark_wf_weights = mark_wf_results["weights"]
mark_wf_window_metrics = mark_wf_results["window_metrics"]
mark_wf_capital_path = mark_wf_results["capital_path"]

# =========================
# Portfolio capital path plot
# =========================

plt.figure(figsize=(16, 9))
plt.plot(
    mark_wf_capital_path["Date"],
    mark_wf_capital_path["Portfolio Value"],
    label="Portfolio Value (Quarterly Rebalanced)"
)
plt.title("Portfolio Evolution - MARK_WF Sample VCV (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Print rolling-window results
# =========================

minimum_display_weight = 0.005

for _, metrics_row in mark_wf_window_metrics.iterrows():
    window_number = int(metrics_row["Rolling Window"])

    print(f"Rolling window {window_number}:")

    weights_row = mark_wf_weights.loc[
        mark_wf_weights["Rolling Window"] == window_number,
        stock_columns
    ].iloc[0]

    weights_table = pd.DataFrame(
        {
            "Stock": stock_columns,
            "Weight": weights_row.values,
        }
    )

    filtered_weights = weights_table[
        weights_table["Weight"] >= minimum_display_weight
    ]

    for _, weight_row in filtered_weights.iterrows():
        print(f"{weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

    print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
    print(f"Annualized return: {metrics_row['annual_return']:.4f}")
    print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
    print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
    print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
    print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
    print("---")

# =========================
# Histograms of rolling-window metrics
# =========================

def plot_metric_histogram(data, title, xlabel):
    plt.figure(figsize=(8, 5))
    plt.hist(pd.Series(data).dropna(), bins=30, alpha=0.7, edgecolor="black")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frequency")
    plt.grid(True)
    plt.show()


# =========================
# Consolidated distribution plot of rolling-window metrics
# =========================

plot_markowitz_metric_distributions(
    mark_wf_window_metrics,
    "MARK_WF"
)

# =========================
# Export results to Excel
# =========================

export_mark_wf_results = True

if export_mark_wf_results:
    mark_wf_filtered_weights = mark_wf_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    mark_wf_filtered_weights = mark_wf_filtered_weights[
        mark_wf_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "MARK_WF.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        mark_wf_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        mark_wf_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        mark_wf_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        mark_wf_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"MARK_WF results exported to {export_file}")

## MARK_WF_RF - Sample Covariance, Risk-Free Rate, No Weight Cap

In [ ]:
# =========================
# MARK_WF_RF - Sample Covariance, Risk-Free Rate, No Weight Cap
# =========================

variant_name = "MARK_WF_RF"

# Run the Markowitz variant.
markowitz_results[variant_name] = run_markowitz_variant(
    variant_name=variant_name,
    covariance_matrices=sample_cov_matrices,
    covariance_matrix_key="Covariance Matrix",
    max_weight=1.0,
    use_risk_free=True,
)

# Extract results for easier use.
mark_wf_rf_results = markowitz_results[variant_name]
mark_wf_rf_weights = mark_wf_rf_results["weights"]
mark_wf_rf_window_metrics = mark_wf_rf_results["window_metrics"]
mark_wf_rf_capital_path = mark_wf_rf_results["capital_path"]

# =========================
# Portfolio capital path plot
# =========================

plt.figure(figsize=(16, 9))
plt.plot(
    mark_wf_rf_capital_path["Date"],
    mark_wf_rf_capital_path["Portfolio Value"],
    label="Portfolio Value (Quarterly Rebalanced)"
)
plt.title("Portfolio Evolution - MARK_WF_RF Sample VCV with Risk-Free Rate (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Print rolling-window results
# =========================

minimum_display_weight = 0.005

for _, metrics_row in mark_wf_rf_window_metrics.iterrows():
    window_number = int(metrics_row["Rolling Window"])

    print(f"Rolling window {window_number}:")
    print(f"Risk-free rate used: {metrics_row['Risk-Free Rate']:.5f}")

    weights_row = mark_wf_rf_weights.loc[
        mark_wf_rf_weights["Rolling Window"] == window_number,
        stock_columns
    ].iloc[0]

    weights_table = pd.DataFrame(
        {
            "Stock": stock_columns,
            "Weight": weights_row.values,
        }
    )

    filtered_weights = weights_table[
        weights_table["Weight"] >= minimum_display_weight
    ]

    for _, weight_row in filtered_weights.iterrows():
        print(f"{weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

    print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
    print(f"Annualized return: {metrics_row['annual_return']:.4f}")
    print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
    print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
    print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
    print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
    print("---")

# =========================
# Consolidated distribution plot of rolling-window metrics
# =========================

plot_markowitz_metric_distributions(
    mark_wf_rf_window_metrics,
    "MARK_WF_RF"
)

# =========================
# Export results to Excel
# =========================

export_mark_wf_rf_results = True

if export_mark_wf_rf_results:
    mark_wf_rf_filtered_weights = mark_wf_rf_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    mark_wf_rf_filtered_weights = mark_wf_rf_filtered_weights[
        mark_wf_rf_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "MARK_WF_RF.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        mark_wf_rf_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        mark_wf_rf_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        mark_wf_rf_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        mark_wf_rf_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"MARK_WF_RF results exported to {export_file}")

## MARK15_S - Sample Covariance, 15% Weight Cap, No Risk-Free Rate

In [ ]:
# =========================
# MARK15_S - Sample Covariance, 15% Weight Cap, No Risk-Free Rate
# =========================

variant_name = "MARK15_S"

# Run the Markowitz variant.
markowitz_results[variant_name] = run_markowitz_variant(
    variant_name=variant_name,
    covariance_matrices=sample_cov_matrices,
    covariance_matrix_key="Covariance Matrix",
    max_weight=0.15,
    use_risk_free=False,
)

# Extract results for easier use.
mark15_s_results = markowitz_results[variant_name]
mark15_s_weights = mark15_s_results["weights"]
mark15_s_window_metrics = mark15_s_results["window_metrics"]
mark15_s_capital_path = mark15_s_results["capital_path"]

# =========================
# Portfolio capital path plot
# =========================

plt.figure(figsize=(16, 9))
plt.plot(
    mark15_s_capital_path["Date"],
    mark15_s_capital_path["Portfolio Value"],
    label="Portfolio Value (Quarterly Rebalanced)"
)
plt.title("Portfolio Evolution - MARK15_S Sample VCV with 15% Weight Cap (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Print rolling-window results
# =========================

minimum_display_weight = 0.005

for _, metrics_row in mark15_s_window_metrics.iterrows():
    window_number = int(metrics_row["Rolling Window"])

    print(f"Rolling window {window_number}:")

    weights_row = mark15_s_weights.loc[
        mark15_s_weights["Rolling Window"] == window_number,
        stock_columns
    ].iloc[0]

    weights_table = pd.DataFrame(
        {
            "Stock": stock_columns,
            "Weight": weights_row.values,
        }
    )

    filtered_weights = weights_table[
        weights_table["Weight"] >= minimum_display_weight
    ]

    for _, weight_row in filtered_weights.iterrows():
        print(f"{weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

    print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
    print(f"Annualized return: {metrics_row['annual_return']:.4f}")
    print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
    print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
    print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
    print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
    print("---")

# =========================
# Consolidated distribution plot of rolling-window metrics
# =========================

plot_markowitz_metric_distributions(
    mark15_s_window_metrics,
    "MARK15_S"
)

# =========================
# Export results to Excel
# =========================

export_mark15_s_results = True

if export_mark15_s_results:
    mark15_s_filtered_weights = mark15_s_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    mark15_s_filtered_weights = mark15_s_filtered_weights[
        mark15_s_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "MARK15_S.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        mark15_s_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        mark15_s_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        mark15_s_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        mark15_s_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"MARK15_S results exported to {export_file}")

## MARK15_S_RF - Sample Covariance, 15% Weight Cap, Risk-Free Rate

In [ ]:
# =========================
# MARK15_S_RF - Sample Covariance, 15% Weight Cap, Risk-Free Rate
# =========================

variant_name = "MARK15_S_RF"

# Run the Markowitz variant.
markowitz_results[variant_name] = run_markowitz_variant(
    variant_name=variant_name,
    covariance_matrices=sample_cov_matrices,
    covariance_matrix_key="Covariance Matrix",
    max_weight=0.15,
    use_risk_free=True,
)

# Extract results for easier use.
mark15_s_rf_results = markowitz_results[variant_name]
mark15_s_rf_weights = mark15_s_rf_results["weights"]
mark15_s_rf_window_metrics = mark15_s_rf_results["window_metrics"]
mark15_s_rf_capital_path = mark15_s_rf_results["capital_path"]

# =========================
# Portfolio capital path plot
# =========================

plt.figure(figsize=(16, 9))
plt.plot(
    mark15_s_rf_capital_path["Date"],
    mark15_s_rf_capital_path["Portfolio Value"],
    label="Portfolio Value (Quarterly Rebalanced)"
)
plt.title("Portfolio Evolution - MARK15_S_RF Sample VCV with 15% Weight Cap and Risk-Free Rate (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Print rolling-window results
# =========================

minimum_display_weight = 0.005

for _, metrics_row in mark15_s_rf_window_metrics.iterrows():
    window_number = int(metrics_row["Rolling Window"])

    print(f"Rolling window {window_number}:")
    print(f"Risk-free rate used: {metrics_row['Risk-Free Rate']:.5f}")

    weights_row = mark15_s_rf_weights.loc[
        mark15_s_rf_weights["Rolling Window"] == window_number,
        stock_columns
    ].iloc[0]

    weights_table = pd.DataFrame(
        {
            "Stock": stock_columns,
            "Weight": weights_row.values,
        }
    )

    filtered_weights = weights_table[
        weights_table["Weight"] >= minimum_display_weight
    ]

    for _, weight_row in filtered_weights.iterrows():
        print(f"{weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

    print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
    print(f"Annualized return: {metrics_row['annual_return']:.4f}")
    print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
    print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
    print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
    print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
    print("---")

# =========================
# Consolidated distribution plot of rolling-window metrics
# =========================

plot_markowitz_metric_distributions(
    mark15_s_rf_window_metrics,
    "MARK15_S_RF"
)

# =========================
# Export results to Excel
# =========================

export_mark15_s_rf_results = True

if export_mark15_s_rf_results:
    mark15_s_rf_filtered_weights = mark15_s_rf_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    mark15_s_rf_filtered_weights = mark15_s_rf_filtered_weights[
        mark15_s_rf_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "MARK15_S_RF.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        mark15_s_rf_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        mark15_s_rf_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        mark15_s_rf_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        mark15_s_rf_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"MARK15_S_RF results exported to {export_file}")

## MARK15_EWMA - EWMA Covariance, 15% Weight Cap, No Risk-Free Rate

In [ ]:
# =========================
# MARK15_EWMA - EWMA Covariance, 15% Weight Cap, No Risk-Free Rate
# =========================

variant_name = "MARK15_EWMA"

# Run the Markowitz variant.
markowitz_results[variant_name] = run_markowitz_variant(
    variant_name=variant_name,
    covariance_matrices=ewma_cov_matrices,
    covariance_matrix_key="EWMA Covariance Matrix",
    max_weight=0.15,
    use_risk_free=False,
)

# Extract results for easier use.
mark15_ewma_results = markowitz_results[variant_name]
mark15_ewma_weights = mark15_ewma_results["weights"]
mark15_ewma_window_metrics = mark15_ewma_results["window_metrics"]
mark15_ewma_capital_path = mark15_ewma_results["capital_path"]

# =========================
# Portfolio capital path plot
# =========================

plt.figure(figsize=(16, 9))
plt.plot(
    mark15_ewma_capital_path["Date"],
    mark15_ewma_capital_path["Portfolio Value"],
    label="Portfolio Value (Quarterly Rebalanced)"
)
plt.title("Portfolio Evolution - MARK15_EWMA EWMA VCV with 15% Weight Cap (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Print rolling-window results
# =========================

minimum_display_weight = 0.005

for _, metrics_row in mark15_ewma_window_metrics.iterrows():
    window_number = int(metrics_row["Rolling Window"])

    print(f"Rolling window {window_number}:")

    weights_row = mark15_ewma_weights.loc[
        mark15_ewma_weights["Rolling Window"] == window_number,
        stock_columns
    ].iloc[0]

    weights_table = pd.DataFrame(
        {
            "Stock": stock_columns,
            "Weight": weights_row.values,
        }
    )

    filtered_weights = weights_table[
        weights_table["Weight"] >= minimum_display_weight
    ]

    for _, weight_row in filtered_weights.iterrows():
        print(f"{weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

    print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
    print(f"Annualized return: {metrics_row['annual_return']:.4f}")
    print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
    print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
    print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
    print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
    print("---")

# =========================
# Consolidated distribution plot of rolling-window metrics
# =========================

plot_markowitz_metric_distributions(
    mark15_ewma_window_metrics,
    "MARK15_EWMA"
)

# =========================
# Export results to Excel
# =========================

export_mark15_ewma_results = True

if export_mark15_ewma_results:
    mark15_ewma_filtered_weights = mark15_ewma_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    mark15_ewma_filtered_weights = mark15_ewma_filtered_weights[
        mark15_ewma_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "MARK15_EWMA.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        mark15_ewma_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        mark15_ewma_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        mark15_ewma_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        mark15_ewma_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"MARK15_EWMA results exported to {export_file}")

## MARK5_EWMA - EWMA Covariance, 5% Weight Cap, No Risk-Free Rate

In [ ]:
# =========================
# MARK5_EWMA - EWMA Covariance, 5% Weight Cap, No Risk-Free Rate
# =========================

variant_name = "MARK5_EWMA"

# Run the Markowitz variant.
markowitz_results[variant_name] = run_markowitz_variant(
    variant_name=variant_name,
    covariance_matrices=ewma_cov_matrices,
    covariance_matrix_key="EWMA Covariance Matrix",
    max_weight=0.05,
    use_risk_free=False,
)

# Extract results for easier use.
mark5_ewma_results = markowitz_results[variant_name]
mark5_ewma_weights = mark5_ewma_results["weights"]
mark5_ewma_window_metrics = mark5_ewma_results["window_metrics"]
mark5_ewma_capital_path = mark5_ewma_results["capital_path"]

# =========================
# Portfolio capital path plot
# =========================

plt.figure(figsize=(16, 9))
plt.plot(
    mark5_ewma_capital_path["Date"],
    mark5_ewma_capital_path["Portfolio Value"],
    label="Portfolio Value (Quarterly Rebalanced)"
)
plt.title("Portfolio Evolution - MARK5_EWMA EWMA VCV with 5% Weight Cap (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Print rolling-window results
# =========================

minimum_display_weight = 0.005

for _, metrics_row in mark5_ewma_window_metrics.iterrows():
    window_number = int(metrics_row["Rolling Window"])

    print(f"Rolling window {window_number}:")

    weights_row = mark5_ewma_weights.loc[
        mark5_ewma_weights["Rolling Window"] == window_number,
        stock_columns
    ].iloc[0]

    weights_table = pd.DataFrame(
        {
            "Stock": stock_columns,
            "Weight": weights_row.values,
        }
    )

    filtered_weights = weights_table[
        weights_table["Weight"] >= minimum_display_weight
    ]

    for _, weight_row in filtered_weights.iterrows():
        print(f"{weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

    print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
    print(f"Annualized return: {metrics_row['annual_return']:.4f}")
    print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
    print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
    print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
    print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
    print("---")

# =========================
# Consolidated distribution plot of rolling-window metrics
# =========================

plot_markowitz_metric_distributions(
    mark5_ewma_window_metrics,
    "MARK5_EWMA"
)

# =========================
# Export results to Excel
# =========================

export_mark5_ewma_results = True

if export_mark5_ewma_results:
    mark5_ewma_filtered_weights = mark5_ewma_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    mark5_ewma_filtered_weights = mark5_ewma_filtered_weights[
        mark5_ewma_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "MARK5_EWMA.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        mark5_ewma_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        mark5_ewma_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        mark5_ewma_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        mark5_ewma_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"MARK5_EWMA results exported to {export_file}")

## Markowitz Results Consolidation and Export

In [ ]:
# =========================
# Markowitz Results Consolidation and Export
# =========================

markowitz_variant_order = [
    "MARK_WF",
    "MARK_WF_RF",
    "MARK15_S",
    "MARK15_S_RF",
    "MARK15_EWMA",
    "MARK5_EWMA",
]

# Check that all Markowitz variants have been computed.
missing_variants = [
    variant for variant in markowitz_variant_order
    if variant not in markowitz_results
]

if missing_variants:
    raise ValueError(
        "The following Markowitz variants are missing from markowitz_results: "
        + ", ".join(missing_variants)
    )

# =========================
# Build wide OOS returns table for future ANALYSIS.xlsx
# =========================

markowitz_oos_returns_df = pd.DataFrame({
    "Rolling Window": range(1, len(rolling_windows) + 1)
})

for variant in markowitz_variant_order:
    variant_metrics = markowitz_results[variant]["window_metrics"]

    markowitz_oos_returns_df[variant] = variant_metrics.sort_values(
        "Rolling Window"
    )["cumulative_return"].values

print("Markowitz OOS returns table:")
print(
    markowitz_oos_returns_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)

# =========================
# Combine all Markowitz window metrics
# =========================

markowitz_window_metrics_all = pd.concat(
    [
        markowitz_results[variant]["window_metrics"]
        for variant in markowitz_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all Markowitz weights
# =========================

markowitz_weights_all = pd.concat(
    [
        markowitz_results[variant]["weights"]
        for variant in markowitz_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all Markowitz capital paths
# =========================

markowitz_capital_paths_all = pd.concat(
    [
        markowitz_results[variant]["capital_path"]
        for variant in markowitz_variant_order
    ],
    ignore_index=True
)

# =========================
# Build compact summary table
# =========================

markowitz_summary_records = []

for variant in markowitz_variant_order:
    variant_metrics = markowitz_results[variant]["window_metrics"]
    variant_capital_path = markowitz_results[variant]["capital_path"]

    final_portfolio_value = variant_capital_path["Portfolio Value"].iloc[-1]
    total_cumulative_return = final_portfolio_value / 100 - 1

    markowitz_summary_records.append(
        {
            "Variant": variant,
            "Number of Rolling Windows": len(variant_metrics),
            "Mean OOS Cumulative Return": variant_metrics["cumulative_return"].mean(),
            "Mean Annual Return": variant_metrics["annual_return"].mean(),
            "Mean Annual Volatility": variant_metrics["annual_volatility"].mean(),
            "Mean Sharpe Ratio": variant_metrics["sharpe_ratio"].mean(),
            "Mean Max Drawdown": variant_metrics["max_drawdown"].mean(),
            "Mean Hit Ratio": variant_metrics["hit_ratio"].mean(),
            "Final Portfolio Value": final_portfolio_value,
            "Total Cumulative Return": total_cumulative_return,
        }
    )

markowitz_summary_df = pd.DataFrame(markowitz_summary_records)

print("\nMarkowitz summary table:")
print(
    markowitz_summary_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)

# =========================
# Consolidated capital path plot
# =========================

plt.figure(figsize=(16, 9))

for variant in markowitz_variant_order:
    variant_path = markowitz_capital_paths_all[
        markowitz_capital_paths_all["Variant"] == variant
    ]

    plt.plot(
        variant_path["Date"],
        variant_path["Portfolio Value"],
        label=variant
    )

plt.title("Portfolio Evolution - Markowitz Variants (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Export consolidated Markowitz results
# =========================

export_consolidated_markowitz_results = False

if export_consolidated_markowitz_results:
    export_file = output_path + "MARKOWITZ_CONSOLIDATED.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        markowitz_oos_returns_df.to_excel(
            writer,
            sheet_name="OOS_Returns",
            index=False
        )

        markowitz_summary_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        markowitz_window_metrics_all.to_excel(
            writer,
            sheet_name="Window_Metrics_All",
            index=False
        )

        markowitz_weights_all.to_excel(
            writer,
            sheet_name="Weights_All",
            index=False
        )

        markowitz_capital_paths_all.to_excel(
            writer,
            sheet_name="Capital_Paths_All",
            index=False
        )

    print(f"Consolidated Markowitz results exported to {export_file}")

# 5. RISK PARITY Portfolio Optimization Variants

## Risk Parity Optimization Framework

In [ ]:
# =========================
# Risk Parity Optimization Framework
# =========================

risk_parity_max_weight = 0.15
risk_parity_top_n = len(stock_columns)
risk_parity_scale_to_target_vol = False
risk_parity_target_vol = 0.15

# Kept consistent with the original thesis code for diagnostic expected volatility.
risk_parity_vol_annualization_factor = 4


def risk_parity_objective(weights, covariance_matrix):
    """
    Risk parity objective based on percentage risk contributions.

    The objective minimizes the squared distance between each asset's
    percentage risk contribution and the equal risk contribution target.
    """
    epsilon = 1e-12

    weights = np.asarray(weights)
    portfolio_variance = float(weights.T @ covariance_matrix @ weights) + epsilon

    marginal_risk = covariance_matrix @ weights
    raw_risk_contributions = weights * marginal_risk
    percentage_risk_contributions = raw_risk_contributions / portfolio_variance

    target_contribution = 1.0 / len(weights)

    return np.sum((percentage_risk_contributions - target_contribution) ** 2)


def compute_percentage_risk_contributions(weights, covariance_matrix):
    """
    Compute percentage risk contributions for a given weight vector.
    """
    portfolio_variance = float(weights.T @ covariance_matrix @ weights)

    if portfolio_variance <= 0:
        portfolio_variance = 1e-12

    raw_risk_contributions = weights * (covariance_matrix @ weights)
    percentage_risk_contributions = raw_risk_contributions / portfolio_variance

    return percentage_risk_contributions


def run_risk_parity_variant(
    variant_name,
    covariance_matrices,
    covariance_matrix_key,
    max_weight=0.15,
    top_n=None,
    scale_to_target_vol=False,
    target_vol=0.15,
):
    """
    Run a Risk Parity optimization variant across all rolling windows.
    """
    if top_n is None:
        top_n = len(stock_columns)

    optimized_weights = []
    window_metrics = []
    capital_series = []
    risk_contribution_records = []
    active_asset_records = []

    current_capital = 100

    n_assets = len(stock_columns)

    initial_weights = np.full(n_assets, 1.0 / n_assets)
    initial_weights = np.minimum(initial_weights, max_weight)

    if initial_weights.sum() == 0:
        initial_weights = np.full(n_assets, 1.0 / n_assets)
    else:
        initial_weights = initial_weights / initial_weights.sum()

    constraints = {
        "type": "eq",
        "fun": lambda weights: np.sum(weights) - 1,
    }

    bounds = [(0, max_weight) for _ in stock_columns]

    for window_index, ((window_start, window_end), covariance_item) in enumerate(
        zip(rolling_windows, covariance_matrices),
        start=1,
    ):
        covariance_matrix = covariance_item[covariance_matrix_key]
        covariance_matrix = covariance_matrix.loc[stock_columns, stock_columns].values

        # Small numerical adjustment for optimizer stability.
        covariance_matrix = covariance_matrix + np.eye(n_assets) * 1e-8

        optimization_result = minimize(
            risk_parity_objective,
            initial_weights,
            args=(covariance_matrix,),
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 1000, "ftol": 1e-12},
        )

        if not optimization_result.success:
            print(
                f"Risk Parity optimization failed for {variant_name}, "
                f"Rolling Window {window_index}, ending {window_end.date()}: "
                f"{optimization_result.message}"
            )

            weights = np.full(n_assets, np.nan)
            active_indices = np.array([], dtype=int)
            percentage_risk_contributions = np.full(n_assets, np.nan)

        else:
            full_weights = optimization_result.x.copy()

            if top_n >= len(full_weights):
                active_indices = np.arange(len(full_weights))
            else:
                active_indices = np.argsort(full_weights)[-top_n:]

            weights = np.zeros_like(full_weights)
            weights[active_indices] = full_weights[active_indices]

            weights = np.minimum(weights, max_weight)

            if weights.sum() <= 0:
                weights = np.zeros_like(full_weights)
                weights[active_indices] = 1.0 / len(active_indices)
            else:
                weights = weights / weights.sum()

            if scale_to_target_vol:
                current_volatility = (
                    np.sqrt(weights.T @ covariance_matrix @ weights)
                    * np.sqrt(risk_parity_vol_annualization_factor)
                )

                if current_volatility > 0:
                    scale_factor = target_vol / current_volatility
                    weights = weights * scale_factor
                    weights = np.minimum(weights, max_weight)

                    if weights.sum() <= 0:
                        weights[active_indices] = 1.0 / len(active_indices)
                    else:
                        weights = weights / weights.sum()

            percentage_risk_contributions = compute_percentage_risk_contributions(
                weights,
                covariance_matrix,
            )

        optimized_weights.append(weights)

        portfolio_variance = (
            weights.T @ covariance_matrix @ weights
            if not np.isnan(weights).all()
            else np.nan
        )

        expected_annual_volatility = (
            np.sqrt(portfolio_variance) * np.sqrt(risk_parity_vol_annualization_factor)
            if pd.notna(portfolio_variance)
            else np.nan
        )

        active_stocks = [stock_columns[i] for i in active_indices]
        active_asset_records.append(
            {
                "Variant": variant_name,
                "Rolling Window": window_index,
                "Active Assets": ", ".join(active_stocks),
                "Number of Active Assets": len(active_stocks),
            }
        )

        for asset_index, stock in enumerate(stock_columns):
            risk_contribution_records.append(
                {
                    "Variant": variant_name,
                    "Rolling Window": window_index,
                    "Stock": stock,
                    "Weight": weights[asset_index],
                    "Risk Contribution": percentage_risk_contributions[asset_index],
                }
            )

        # Out-of-sample evaluation.
        oos_start = window_end + pd.Timedelta(days=1)
        oos_end = oos_start + pd.DateOffset(months=rebalancing_period_months) - pd.Timedelta(days=1)

        if oos_end > stocks_df.index[-1]:
            oos_end = stocks_df.index[-1]

        oos_prices = stocks_df.loc[oos_start:oos_end]
        oos_returns = compute_simple_returns(oos_prices)

        if oos_returns.empty or np.isnan(weights).all():
            portfolio_returns = pd.Series(dtype=float)
        else:
            portfolio_returns = oos_returns.dot(weights)

        metrics = compute_oos_metrics(
            portfolio_returns,
            risk_free_rate=0.0,
            use_risk_free=False,
        )

        window_metrics.append(
            {
                "Variant": variant_name,
                "Rolling Window": window_index,
                "Lookback Start": window_start.strftime("%Y-%m-%d"),
                "Lookback End": window_end.strftime("%Y-%m-%d"),
                "OOS Start": oos_prices.index.min().strftime("%Y-%m-%d")
                if not oos_prices.empty
                else None,
                "OOS End": oos_prices.index.max().strftime("%Y-%m-%d")
                if not oos_prices.empty
                else None,
                "Number of Active Assets": len(active_stocks),
                "Expected Annual Volatility": expected_annual_volatility,
                "Optimization Success": optimization_result.success,
                "cumulative_return": metrics["cumulative_return"],
                "annual_return": metrics["annual_return"],
                "annual_volatility": metrics["annual_volatility"],
                "sharpe_ratio": metrics["sharpe_ratio"],
                "max_drawdown": metrics["max_drawdown"],
                "hit_ratio": metrics["hit_ratio"],
            }
        )

        if portfolio_returns.empty:
            for date in oos_prices.index:
                capital_series.append(
                    {
                        "Date": date,
                        "Variant": variant_name,
                        "Portfolio Value": current_capital,
                    }
                )
        else:
            for date, portfolio_return in portfolio_returns.items():
                current_capital *= 1 + portfolio_return

                capital_series.append(
                    {
                        "Date": date,
                        "Variant": variant_name,
                        "Portfolio Value": current_capital,
                    }
                )

    weights_df = pd.DataFrame(
        optimized_weights,
        columns=stock_columns,
    )
    weights_df.insert(0, "Rolling Window", range(1, len(weights_df) + 1))
    weights_df.insert(0, "Variant", variant_name)

    return {
        "variant_name": variant_name,
        "weights": weights_df,
        "window_metrics": pd.DataFrame(window_metrics),
        "capital_path": pd.DataFrame(capital_series),
        "risk_contributions": pd.DataFrame(risk_contribution_records),
        "active_assets": pd.DataFrame(active_asset_records),
    }


def plot_risk_parity_metric_distributions(window_metrics_df, variant_name):
    """
    Plot rolling-window metric distributions in a single consolidated figure.
    """
    metric_specs = [
        ("cumulative_return", "Cumulative Return"),
        ("annual_return", "Annualized Return"),
        ("annual_volatility", "Annualized Volatility"),
        ("sharpe_ratio", "Sharpe Ratio"),
        ("max_drawdown", "Max Drawdown"),
        ("hit_ratio", "Hit Ratio"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for ax, (metric_column, metric_label) in zip(axes, metric_specs):
        ax.hist(
            window_metrics_df[metric_column].dropna(),
            bins=30,
            alpha=0.7,
            edgecolor="black"
        )
        ax.set_title(metric_label)
        ax.set_xlabel(metric_label)
        ax.set_ylabel("Frequency")
        ax.grid(True)

    fig.suptitle(
        f"Distribution of Rolling-Window Performance Metrics - {variant_name}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


def report_and_export_risk_parity_variant(result, export_results=True):
    """
    Print, plot, and export a Risk Parity variant result.
    """
    variant_name = result["variant_name"]
    weights_df = result["weights"]
    window_metrics_df = result["window_metrics"]
    capital_path_df = result["capital_path"]
    risk_contributions_df = result["risk_contributions"]
    active_assets_df = result["active_assets"]

    # Portfolio path plot.
    plt.figure(figsize=(16, 9))
    plt.plot(
        capital_path_df["Date"],
        capital_path_df["Portfolio Value"],
        label="Portfolio Value (Risk Parity Quarterly Rebalanced)"
    )
    plt.title(f"Portfolio Evolution - {variant_name} (2018-2024)")
    plt.xlabel("Date")
    plt.ylabel("Portfolio Value")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Print rolling-window results.
    minimum_display_weight = 0.005

    for _, metrics_row in window_metrics_df.iterrows():
        window_number = int(metrics_row["Rolling Window"])

        print(
            f"Rolling window {window_number} "
            f"(end {metrics_row['Lookback End']}):"
        )
        print(
            f"Active assets: {int(metrics_row['Number of Active Assets'])} | "
            f"Expected annual vol: {metrics_row['Expected Annual Volatility']:.2%}"
        )

        weights_row = weights_df.loc[
            weights_df["Rolling Window"] == window_number,
            stock_columns
        ].iloc[0]

        weights_table = pd.DataFrame(
            {
                "Stock": stock_columns,
                "Weight": weights_row.values,
            }
        )

        filtered_weights = weights_table[
            weights_table["Weight"] >= minimum_display_weight
        ]

        print("Active stocks & weights:")
        for _, weight_row in filtered_weights.iterrows():
            print(f"  {weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

        filtered_risk_contributions = risk_contributions_df[
            (risk_contributions_df["Rolling Window"] == window_number)
            & (risk_contributions_df["Weight"] >= minimum_display_weight)
        ]

        print("Percentage risk contributions (approx):")
        for _, rc_row in filtered_risk_contributions.iterrows():
            print(f"  {rc_row['Stock']}: {rc_row['Risk Contribution'] * 100:.2f}%")

        print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
        print(f"Annualized return: {metrics_row['annual_return']:.4f}")
        print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
        print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
        print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
        print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
        print("---")

    # Consolidated histograms.
    plot_risk_parity_metric_distributions(
        window_metrics_df,
        variant_name
    )

    

## RP_S - Sample Covariance Risk Parity

In [ ]:
# =========================
# RP_S - Sample Covariance Risk Parity
# =========================

variant_name = "RP_S"

try:
    risk_parity_results
except NameError:
    risk_parity_results = {}

# Run the Risk Parity variant.
risk_parity_results[variant_name] = run_risk_parity_variant(
    variant_name=variant_name,
    covariance_matrices=sample_cov_matrices,
    covariance_matrix_key="Covariance Matrix",
    max_weight=risk_parity_max_weight,
    top_n=risk_parity_top_n,
    scale_to_target_vol=risk_parity_scale_to_target_vol,
    target_vol=risk_parity_target_vol,
)

# Extract results for easier use.
rp_s_results = risk_parity_results[variant_name]
rp_s_weights = rp_s_results["weights"]
rp_s_window_metrics = rp_s_results["window_metrics"]
rp_s_capital_path = rp_s_results["capital_path"]
rp_s_risk_contributions = rp_s_results["risk_contributions"]
rp_s_active_assets = rp_s_results["active_assets"]

# Print results and produce diagnostic plots.
report_and_export_risk_parity_variant(
    rp_s_results
)

# =========================
# Export results to Excel
# =========================

export_rp_s_results = True

if export_rp_s_results:
    rp_s_filtered_weights = rp_s_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    rp_s_filtered_weights = rp_s_filtered_weights[
        rp_s_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "RP_S.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        rp_s_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        rp_s_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        rp_s_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        rp_s_risk_contributions.to_excel(
            writer,
            sheet_name="Risk_Contributions",
            index=False
        )

        rp_s_active_assets.to_excel(
            writer,
            sheet_name="Active_Assets",
            index=False
        )

        rp_s_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"RP_S results exported to {export_file}")

## RP_EWMA - EWMA Covariance Risk Parity

In [ ]:
# =========================
# RP_EWMA - EWMA Covariance Risk Parity
# =========================

variant_name = "RP_EWMA"

# Run the Risk Parity variant.
risk_parity_results[variant_name] = run_risk_parity_variant(
    variant_name=variant_name,
    covariance_matrices=ewma_cov_matrices,
    covariance_matrix_key="EWMA Covariance Matrix",
    max_weight=risk_parity_max_weight,
    top_n=risk_parity_top_n,
    scale_to_target_vol=risk_parity_scale_to_target_vol,
    target_vol=risk_parity_target_vol,
)

# Extract results for easier use.
rp_ewma_results = risk_parity_results[variant_name]
rp_ewma_weights = rp_ewma_results["weights"]
rp_ewma_window_metrics = rp_ewma_results["window_metrics"]
rp_ewma_capital_path = rp_ewma_results["capital_path"]
rp_ewma_risk_contributions = rp_ewma_results["risk_contributions"]
rp_ewma_active_assets = rp_ewma_results["active_assets"]

# Print results and produce diagnostic plots.
report_and_export_risk_parity_variant(
    rp_ewma_results
)

# =========================
# Export results to Excel
# =========================

export_rp_ewma_results = True

if export_rp_ewma_results:
    rp_ewma_filtered_weights = rp_ewma_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    rp_ewma_filtered_weights = rp_ewma_filtered_weights[
        rp_ewma_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "RP_EWMA.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        rp_ewma_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        rp_ewma_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        rp_ewma_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        rp_ewma_risk_contributions.to_excel(
            writer,
            sheet_name="Risk_Contributions",
            index=False
        )

        rp_ewma_active_assets.to_excel(
            writer,
            sheet_name="Active_Assets",
            index=False
        )

        rp_ewma_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"RP_EWMA results exported to {export_file}")

## Risk Parity Results Consolidation and Export

In [ ]:
# =========================
# Risk Parity Results Consolidation and Export
# =========================

risk_parity_variant_order = [
    "RP_S",
    "RP_EWMA",
]

# Check that all Risk Parity variants have been computed.
missing_variants = [
    variant for variant in risk_parity_variant_order
    if variant not in risk_parity_results
]

if missing_variants:
    raise ValueError(
        "The following Risk Parity variants are missing from risk_parity_results: "
        + ", ".join(missing_variants)
    )

# =========================
# Build wide OOS returns table for future ANALYSIS.xlsx
# =========================

risk_parity_oos_returns_df = pd.DataFrame({
    "Rolling Window": range(1, len(rolling_windows) + 1)
})

for variant in risk_parity_variant_order:
    variant_metrics = risk_parity_results[variant]["window_metrics"]

    risk_parity_oos_returns_df[variant] = variant_metrics.sort_values(
        "Rolling Window"
    )["cumulative_return"].values

print("Risk Parity OOS returns table:")
print(
    risk_parity_oos_returns_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)

# =========================
# Combine all Risk Parity window metrics
# =========================

risk_parity_window_metrics_all = pd.concat(
    [
        risk_parity_results[variant]["window_metrics"]
        for variant in risk_parity_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all Risk Parity weights
# =========================

risk_parity_weights_all = pd.concat(
    [
        risk_parity_results[variant]["weights"]
        for variant in risk_parity_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all Risk Parity risk contributions
# =========================

risk_parity_risk_contributions_all = pd.concat(
    [
        risk_parity_results[variant]["risk_contributions"]
        for variant in risk_parity_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all Risk Parity active assets
# =========================

risk_parity_active_assets_all = pd.concat(
    [
        risk_parity_results[variant]["active_assets"]
        for variant in risk_parity_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all Risk Parity capital paths
# =========================

risk_parity_capital_paths_all = pd.concat(
    [
        risk_parity_results[variant]["capital_path"]
        for variant in risk_parity_variant_order
    ],
    ignore_index=True
)

# =========================
# Build compact summary table
# =========================

risk_parity_summary_records = []

for variant in risk_parity_variant_order:
    variant_metrics = risk_parity_results[variant]["window_metrics"]
    variant_capital_path = risk_parity_results[variant]["capital_path"]

    final_portfolio_value = variant_capital_path["Portfolio Value"].iloc[-1]
    total_cumulative_return = final_portfolio_value / 100 - 1

    risk_parity_summary_records.append(
        {
            "Variant": variant,
            "Number of Rolling Windows": len(variant_metrics),
            "Mean OOS Cumulative Return": variant_metrics["cumulative_return"].mean(),
            "Mean Annual Return": variant_metrics["annual_return"].mean(),
            "Mean Annual Volatility": variant_metrics["annual_volatility"].mean(),
            "Mean Sharpe Ratio": variant_metrics["sharpe_ratio"].mean(),
            "Mean Max Drawdown": variant_metrics["max_drawdown"].mean(),
            "Mean Hit Ratio": variant_metrics["hit_ratio"].mean(),
            "Final Portfolio Value": final_portfolio_value,
            "Total Cumulative Return": total_cumulative_return,
        }
    )

risk_parity_summary_df = pd.DataFrame(risk_parity_summary_records)

print("\nRisk Parity summary table:")
print(
    risk_parity_summary_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)

# =========================
# Consolidated capital path plot
# =========================

plt.figure(figsize=(16, 9))

for variant in risk_parity_variant_order:
    variant_path = risk_parity_capital_paths_all[
        risk_parity_capital_paths_all["Variant"] == variant
    ]

    plt.plot(
        variant_path["Date"],
        variant_path["Portfolio Value"],
        label=variant
    )

plt.title("Portfolio Evolution - Risk Parity Variants (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Export consolidated Risk Parity results
# =========================

export_consolidated_risk_parity_results = False

if export_consolidated_risk_parity_results:
    export_file = output_path + "RISK_PARITY_CONSOLIDATED.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        risk_parity_oos_returns_df.to_excel(
            writer,
            sheet_name="OOS_Returns",
            index=False
        )

        risk_parity_summary_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        risk_parity_window_metrics_all.to_excel(
            writer,
            sheet_name="Window_Metrics_All",
            index=False
        )

        risk_parity_weights_all.to_excel(
            writer,
            sheet_name="Weights_All",
            index=False
        )

        risk_parity_risk_contributions_all.to_excel(
            writer,
            sheet_name="Risk_Contributions_All",
            index=False
        )

        risk_parity_active_assets_all.to_excel(
            writer,
            sheet_name="Active_Assets_All",
            index=False
        )

        risk_parity_capital_paths_all.to_excel(
            writer,
            sheet_name="Capital_Paths_All",
            index=False
        )

    print(f"Consolidated Risk Parity results exported to {export_file}")

# 6. CVaR Portfolio Optimization Variants

## CVaR Optimization Framework

In [ ]:
# =========================
# CVaR Optimization Framework
# =========================

def optimize_cvar_weights(
    returns,
    confidence_level,
    max_weight,
    long_only=True,
):
    """
    Solve a CVaR minimization problem for a given return matrix.

    Parameters
    ----------
    returns : numpy.ndarray
        Return matrix with observations as rows and assets as columns.

    confidence_level : float
        CVaR confidence level, e.g. 0.95 or 0.99.

    max_weight : float
        Maximum allowed portfolio weight per asset.

    long_only : bool
        If True, short selling is not allowed.

    Returns
    -------
    numpy.ndarray
        Optimized portfolio weights.
    """
    n_observations, n_assets = returns.shape

    weights = cp.Variable(n_assets)
    value_at_risk = cp.Variable()
    tail_losses = cp.Variable(n_observations)

    portfolio_returns = returns @ weights

    constraints = []

    if long_only:
        constraints.append(weights >= 0)

    constraints.append(cp.sum(weights) == 1)
    constraints.append(weights <= max_weight)

    constraints.append(tail_losses >= 0)
    constraints.append(tail_losses + portfolio_returns + value_at_risk >= 0)

    objective = cp.Minimize(
        value_at_risk
        + (1 / (1 - confidence_level))
        * cp.sum(tail_losses)
        / n_observations
    )

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    if weights.value is None:
        raise ValueError("CVaR optimization failed.")

    optimized_weights = np.array(weights.value).flatten()

    # Numerical cleanup for very small solver tolerances.
    optimized_weights[np.abs(optimized_weights) < 1e-10] = 0
    optimized_weights = np.clip(optimized_weights, 0, max_weight)

    if optimized_weights.sum() > 0:
        optimized_weights = optimized_weights / optimized_weights.sum()
    else:
        raise ValueError("CVaR optimization returned zero weights.")

    return optimized_weights


def run_cvar_variant(
    variant_name,
    confidence_level,
    max_weight,
):
    """
    Run a CVaR optimization variant across all rolling windows.
    """
    optimized_weights = []
    window_metrics = []
    capital_series = []

    current_capital = 100

    for window_index, (window_start, window_end) in enumerate(rolling_windows, start=1):
        window_prices = stocks_df.loc[window_start:window_end]
        log_returns = compute_log_returns(window_prices).values

        try:
            weights = optimize_cvar_weights(
                returns=log_returns,
                confidence_level=confidence_level,
                max_weight=max_weight,
                long_only=True,
            )

            optimization_success = True
            optimization_message = "success"

        except Exception as error:
            print(
                f"CVaR optimization failed for {variant_name}, "
                f"Rolling Window {window_index}, ending {window_end.date()}: {error}"
            )

            weights = np.full(len(stock_columns), np.nan)
            optimization_success = False
            optimization_message = str(error)

        optimized_weights.append(weights)

        # Out-of-sample evaluation.
        oos_start = window_end + pd.Timedelta(days=1)
        oos_end = oos_start + pd.DateOffset(months=rebalancing_period_months) - pd.Timedelta(days=1)

        if oos_end > stocks_df.index[-1]:
            oos_end = stocks_df.index[-1]

        oos_prices = stocks_df.loc[oos_start:oos_end]
        oos_returns = compute_simple_returns(oos_prices)

        if oos_returns.empty or np.isnan(weights).all():
            portfolio_returns = pd.Series(dtype=float)
        else:
            portfolio_returns = oos_returns.dot(weights)

        metrics = compute_oos_metrics(
            portfolio_returns,
            risk_free_rate=0.0,
            use_risk_free=False,
        )

        window_metrics.append(
            {
                "Variant": variant_name,
                "Rolling Window": window_index,
                "Lookback Start": window_start.strftime("%Y-%m-%d"),
                "Lookback End": window_end.strftime("%Y-%m-%d"),
                "OOS Start": oos_prices.index.min().strftime("%Y-%m-%d")
                if not oos_prices.empty
                else None,
                "OOS End": oos_prices.index.max().strftime("%Y-%m-%d")
                if not oos_prices.empty
                else None,
                "Confidence Level": confidence_level,
                "Max Weight": max_weight,
                "Optimization Success": optimization_success,
                "Optimization Message": optimization_message,
                "cumulative_return": metrics["cumulative_return"],
                "annual_return": metrics["annual_return"],
                "annual_volatility": metrics["annual_volatility"],
                "sharpe_ratio": metrics["sharpe_ratio"],
                "max_drawdown": metrics["max_drawdown"],
                "hit_ratio": metrics["hit_ratio"],
            }
        )

        if portfolio_returns.empty:
            for date in oos_prices.index:
                capital_series.append(
                    {
                        "Date": date,
                        "Variant": variant_name,
                        "Portfolio Value": current_capital,
                    }
                )
        else:
            for date, portfolio_return in portfolio_returns.items():
                current_capital *= 1 + portfolio_return

                capital_series.append(
                    {
                        "Date": date,
                        "Variant": variant_name,
                        "Portfolio Value": current_capital,
                    }
                )

    weights_df = pd.DataFrame(
        optimized_weights,
        columns=stock_columns,
    )
    weights_df.insert(0, "Rolling Window", range(1, len(weights_df) + 1))
    weights_df.insert(0, "Variant", variant_name)

    return {
        "variant_name": variant_name,
        "weights": weights_df,
        "window_metrics": pd.DataFrame(window_metrics),
        "capital_path": pd.DataFrame(capital_series),
    }


def plot_cvar_metric_distributions(window_metrics_df, variant_name):
    """
    Plot rolling-window CVaR performance metric distributions.
    """
    metric_specs = [
        ("cumulative_return", "Cumulative Return"),
        ("annual_return", "Annualized Return"),
        ("annual_volatility", "Annualized Volatility"),
        ("sharpe_ratio", "Sharpe Ratio"),
        ("max_drawdown", "Max Drawdown"),
        ("hit_ratio", "Hit Ratio"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()

    for ax, (metric_column, metric_label) in zip(axes, metric_specs):
        ax.hist(
            window_metrics_df[metric_column].dropna(),
            bins=30,
            alpha=0.7,
            edgecolor="black"
        )
        ax.set_title(metric_label)
        ax.set_xlabel(metric_label)
        ax.set_ylabel("Frequency")
        ax.grid(True)

    fig.suptitle(
        f"Distribution of Rolling-Window Performance Metrics - {variant_name}",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


def report_cvar_variant(result):
    """
    Print and plot a CVaR variant result.
    """
    variant_name = result["variant_name"]
    weights_df = result["weights"]
    window_metrics_df = result["window_metrics"]
    capital_path_df = result["capital_path"]

    # Portfolio path plot.
    plt.figure(figsize=(16, 9))
    plt.plot(
        capital_path_df["Date"],
        capital_path_df["Portfolio Value"],
        label="Portfolio Value (Quarterly Rebalanced)"
    )
    plt.title(f"Portfolio Evolution - {variant_name} (2018-2024)")
    plt.xlabel("Date")
    plt.ylabel("Portfolio Value")
    plt.legend()
    plt.grid(True)
    plt.show()

    # Print rolling-window results.
    minimum_display_weight = 0.005

    for _, metrics_row in window_metrics_df.iterrows():
        window_number = int(metrics_row["Rolling Window"])

        print(f"Rolling window {window_number}:")
        print(f"Confidence level: {metrics_row['Confidence Level']:.2%}")
        print(f"Max weight: {metrics_row['Max Weight']:.2%}")

        weights_row = weights_df.loc[
            weights_df["Rolling Window"] == window_number,
            stock_columns
        ].iloc[0]

        weights_table = pd.DataFrame(
            {
                "Stock": stock_columns,
                "Weight": weights_row.values,
            }
        )

        filtered_weights = weights_table[
            weights_table["Weight"] >= minimum_display_weight
        ]

        for _, weight_row in filtered_weights.iterrows():
            print(f"{weight_row['Stock']}: {weight_row['Weight'] * 100:.2f}%")

        print(f"Cumulative return next quarter: {metrics_row['cumulative_return']:.4f}")
        print(f"Annualized return: {metrics_row['annual_return']:.4f}")
        print(f"Annualized volatility: {metrics_row['annual_volatility']:.4f}")
        print(f"Sharpe ratio: {metrics_row['sharpe_ratio']:.4f}")
        print(f"Max drawdown: {metrics_row['max_drawdown']:.4f}")
        print(f"Hit ratio: {metrics_row['hit_ratio']:.4f}")
        print("---")

    # Consolidated histograms.
    plot_cvar_metric_distributions(
        window_metrics_df,
        variant_name
    )

## CVAR 95-10 - 95% Confidence Level, 10% Weight Cap

In [ ]:
# =========================
# CVAR 95-10 - 95% Confidence Level, 10% Weight Cap
# =========================

variant_name = "CVAR 95-10"

try:
    cvar_results
except NameError:
    cvar_results = {}

# Run the CVaR variant.
cvar_results[variant_name] = run_cvar_variant(
    variant_name=variant_name,
    confidence_level=0.95,
    max_weight=0.10,
)

# Extract results for easier use.
cvar_95_10_results = cvar_results[variant_name]
cvar_95_10_weights = cvar_95_10_results["weights"]
cvar_95_10_window_metrics = cvar_95_10_results["window_metrics"]
cvar_95_10_capital_path = cvar_95_10_results["capital_path"]

# Print results and produce diagnostic plots.
report_cvar_variant(
    cvar_95_10_results
)

# =========================
# Export results to Excel
# =========================

export_cvar_95_10_results = True

if export_cvar_95_10_results:
    minimum_display_weight = 0.005

    cvar_95_10_filtered_weights = cvar_95_10_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    cvar_95_10_filtered_weights = cvar_95_10_filtered_weights[
        cvar_95_10_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "CVAR_95_10.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        cvar_95_10_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        cvar_95_10_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        cvar_95_10_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        cvar_95_10_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"CVAR 95-10 results exported to {export_file}")

## CVAR 99-15 - 99% Confidence Level, 15% Weight Cap

In [ ]:
# =========================
# CVAR 99-15 - 99% Confidence Level, 15% Weight Cap
# =========================

variant_name = "CVAR 99-15"

# Run the CVaR variant.
cvar_results[variant_name] = run_cvar_variant(
    variant_name=variant_name,
    confidence_level=0.99,
    max_weight=0.15,
)

# Extract results for easier use.
cvar_99_15_results = cvar_results[variant_name]
cvar_99_15_weights = cvar_99_15_results["weights"]
cvar_99_15_window_metrics = cvar_99_15_results["window_metrics"]
cvar_99_15_capital_path = cvar_99_15_results["capital_path"]

# Print results and produce diagnostic plots.
report_cvar_variant(
    cvar_99_15_results
)

# =========================
# Export results to Excel
# =========================

export_cvar_99_15_results = True

if export_cvar_99_15_results:
    minimum_display_weight = 0.005

    cvar_99_15_filtered_weights = cvar_99_15_weights.melt(
        id_vars=["Variant", "Rolling Window"],
        var_name="Stock",
        value_name="Weight"
    )

    cvar_99_15_filtered_weights = cvar_99_15_filtered_weights[
        cvar_99_15_filtered_weights["Weight"] >= minimum_display_weight
    ]

    export_file = output_path + "CVAR_99_15.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        cvar_99_15_window_metrics.to_excel(
            writer,
            sheet_name="Window_Metrics",
            index=False
        )

        cvar_99_15_weights.to_excel(
            writer,
            sheet_name="Weights",
            index=False
        )

        cvar_99_15_filtered_weights.to_excel(
            writer,
            sheet_name="Filtered_Weights",
            index=False
        )

        cvar_99_15_capital_path.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

    print(f"CVAR 99-15 results exported to {export_file}")

## CVaR Results Consolidation and Export

In [ ]:
# =========================
# CVaR Results Consolidation and Export
# =========================

cvar_variant_order = [
    "CVAR 95-10",
    "CVAR 99-15",
]

# Check that all CVaR variants have been computed.
missing_variants = [
    variant for variant in cvar_variant_order
    if variant not in cvar_results
]

if missing_variants:
    raise ValueError(
        "The following CVaR variants are missing from cvar_results: "
        + ", ".join(missing_variants)
    )

# =========================
# Build wide OOS returns table for future ANALYSIS.xlsx
# =========================

cvar_oos_returns_df = pd.DataFrame({
    "Rolling Window": range(1, len(rolling_windows) + 1)
})

for variant in cvar_variant_order:
    variant_metrics = cvar_results[variant]["window_metrics"]

    cvar_oos_returns_df[variant] = variant_metrics.sort_values(
        "Rolling Window"
    )["cumulative_return"].values

print("CVaR OOS returns table:")
print(
    cvar_oos_returns_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)

# =========================
# Combine all CVaR window metrics
# =========================

cvar_window_metrics_all = pd.concat(
    [
        cvar_results[variant]["window_metrics"]
        for variant in cvar_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all CVaR weights
# =========================

cvar_weights_all = pd.concat(
    [
        cvar_results[variant]["weights"]
        for variant in cvar_variant_order
    ],
    ignore_index=True
)

# =========================
# Combine all CVaR capital paths
# =========================

cvar_capital_paths_all = pd.concat(
    [
        cvar_results[variant]["capital_path"]
        for variant in cvar_variant_order
    ],
    ignore_index=True
)

# =========================
# Build compact summary table
# =========================

cvar_summary_records = []

for variant in cvar_variant_order:
    variant_metrics = cvar_results[variant]["window_metrics"]
    variant_capital_path = cvar_results[variant]["capital_path"]

    final_portfolio_value = variant_capital_path["Portfolio Value"].iloc[-1]
    total_cumulative_return = final_portfolio_value / 100 - 1

    cvar_summary_records.append(
        {
            "Variant": variant,
            "Number of Rolling Windows": len(variant_metrics),
            "Mean OOS Cumulative Return": variant_metrics["cumulative_return"].mean(),
            "Mean Annual Return": variant_metrics["annual_return"].mean(),
            "Mean Annual Volatility": variant_metrics["annual_volatility"].mean(),
            "Mean Sharpe Ratio": variant_metrics["sharpe_ratio"].mean(),
            "Mean Max Drawdown": variant_metrics["max_drawdown"].mean(),
            "Mean Hit Ratio": variant_metrics["hit_ratio"].mean(),
            "Final Portfolio Value": final_portfolio_value,
            "Total Cumulative Return": total_cumulative_return,
        }
    )

cvar_summary_df = pd.DataFrame(cvar_summary_records)

print("\nCVaR summary table:")
print(
    cvar_summary_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)

# =========================
# Consolidated capital path plot
# =========================

plt.figure(figsize=(16, 9))

for variant in cvar_variant_order:
    variant_path = cvar_capital_paths_all[
        cvar_capital_paths_all["Variant"] == variant
    ]

    plt.plot(
        variant_path["Date"],
        variant_path["Portfolio Value"],
        label=variant
    )

plt.title("Portfolio Evolution - CVaR Variants (2018-2024)")
plt.xlabel("Date")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.show()

# =========================
# Export consolidated CVaR results
# =========================

export_consolidated_cvar_results = False

if export_consolidated_cvar_results:
    export_file = output_path + "CVAR_CONSOLIDATED.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        cvar_oos_returns_df.to_excel(
            writer,
            sheet_name="OOS_Returns",
            index=False
        )

        cvar_summary_df.to_excel(
            writer,
            sheet_name="Summary",
            index=False
        )

        cvar_window_metrics_all.to_excel(
            writer,
            sheet_name="Window_Metrics_All",
            index=False
        )

        cvar_weights_all.to_excel(
            writer,
            sheet_name="Weights_All",
            index=False
        )

        cvar_capital_paths_all.to_excel(
            writer,
            sheet_name="Capital_Paths_All",
            index=False
        )

    print(f"Consolidated CVaR results exported to {export_file}")

# 7. FINAL ANALYSIS DATASET AND BASELINE EVALUATION

## Build Final ANALYSIS Dataset

In [ ]:
# =========================
# Build Final ANALYSIS Dataset
# =========================

# This cell builds the final analysis_df by combining:
# - market environment variables,
# - regime assignments,
# - benchmark OOS returns,
# - Markowitz OOS returns,
# - Risk Parity OOS returns,
# - CVaR OOS returns.

# =========================
# Required object checks
# =========================

required_objects = [
    "rolling_windows",
    "markowitz_oos_returns_df",
    "risk_parity_oos_returns_df",
    "cvar_oos_returns_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Fallback setup
# =========================

try:
    output_path
except NameError:
    output_path = "output/"

try:
    data_path
except NameError:
    data_path = "FDATA.xlsx"

try:
    rebalancing_period_months
except NameError:
    rebalancing_period_months = 3

try:
    df_prices_full
except NameError:
    df_prices_full = pd.read_excel(
        data_path,
        sheet_name="RDATA",
        parse_dates=["DATE"]
    )
    df_prices_full.set_index("DATE", inplace=True)

try:
    stocks_df
except NameError:
    stock_columns = df_prices_full.columns[1:]
    stocks_df = df_prices_full.loc[:, stock_columns]

try:
    signals_source_df = signals_df.copy()
except NameError:
    signals_source_df = pd.read_excel(output_path + "SIGNALS.xlsx")

if "Rolling Window" in signals_source_df.columns:
    signals_source_df = signals_source_df.sort_values("Rolling Window")

signals_source_df = signals_source_df.reset_index(drop=True)


# =========================
# Helper function for flexible column selection
# =========================

def get_first_available_column(dataframe, candidate_columns, output_name):
    """
    Return the first available column from a list of candidate names.
    """
    for column in candidate_columns:
        if column in dataframe.columns:
            return dataframe[column]

    raise KeyError(
        f"None of the candidate columns for '{output_name}' were found: "
        + ", ".join(candidate_columns)
    )


# =========================
# Build base market environment dataframe
# =========================

analysis_base_df = pd.DataFrame({
    "Rolling Window": range(1, len(rolling_windows) + 1)
})

analysis_base_df["Market Volatility"] = get_first_available_column(
    signals_source_df,
    ["Market Volatility", "Volatility"],
    "Market Volatility"
).iloc[:len(rolling_windows)].values

analysis_base_df["Av.Cr.-Corr."] = get_first_available_column(
    signals_source_df,
    ["Av.Cr.-Corr.", "Average Cross-Correlation", "Average Cross Correlation"],
    "Av.Cr.-Corr."
).iloc[:len(rolling_windows)].values

analysis_base_df["Market Trend"] = get_first_available_column(
    signals_source_df,
    ["Market Trend", "Trend"],
    "Market Trend"
).iloc[:len(rolling_windows)].values

analysis_base_df["Skewness"] = get_first_available_column(
    signals_source_df,
    ["Skewness", "Skew"],
    "Skewness"
).iloc[:len(rolling_windows)].values

analysis_base_df["Kurtosis"] = get_first_available_column(
    signals_source_df,
    ["Kurtosis", "Kurt"],
    "Kurtosis"
).iloc[:len(rolling_windows)].values


# =========================
# Load regime assignments
# =========================

def get_regime_series():
    """
    Load regime assignments from the available data sources.
    """
    regime_candidate_columns = [
    "regime",
    "Regime",
    "Regime_GMM",
    "GMM Regime",
    "GMM_Regime",
    "GMM regime",
    "gmm_regime",
    "cluster",
    "Cluster",
    "GMM Cluster",
    "GMM_Cluster",
    "Market Regime",
    "Market_Regime",
]

    source_dataframes = [
        ("signals_source_df", signals_source_df),
    ]

    try:
        oos_regimes_df = pd.read_excel(
            output_path + "OOS_Net_Returns_with_Regimes.xlsx"
        )
        source_dataframes.append(
            ("OOS_Net_Returns_with_Regimes", oos_regimes_df)
        )
    except FileNotFoundError:
        pass

    try:
        market_regimes_df = pd.read_excel(
            output_path + "Market_Regimes_GMM_with_Probabilities.xlsx"
        )
        source_dataframes.append(
            ("Market_Regimes_GMM_with_Probabilities", market_regimes_df)
        )
    except FileNotFoundError:
        pass

    try:
        market_regimes_assigned_df = pd.read_excel(
            output_path + "Market_Regimes_Assigned.xlsx"
        )
        source_dataframes.append(
            ("Market_Regimes_Assigned", market_regimes_assigned_df)
        )
    except FileNotFoundError:
        pass

    for source_name, source_df in source_dataframes:
        source_df = source_df.copy()

        if "Rolling Window" in source_df.columns:
            source_df = source_df.sort_values("Rolling Window")

        source_df = source_df.reset_index(drop=True)

        for column in regime_candidate_columns:
            if column in source_df.columns:
                regime_series = source_df[column].reset_index(drop=True)

                if len(regime_series) < len(rolling_windows):
                    raise ValueError(
                        f"Regime source '{source_name}' has fewer rows than rolling_windows."
                    )

                return regime_series.iloc[:len(rolling_windows)]

    available_columns_report = {}

    for source_name, source_df in source_dataframes:
        available_columns_report[source_name] = list(source_df.columns)

    raise KeyError(
        "No regime column was found in the available sources. "
        f"Available columns: {available_columns_report}"
    )


analysis_base_df["regime"] = get_regime_series().values
analysis_base_df["regime"] = analysis_base_df["regime"].astype(int)


# =========================
# Build benchmark OOS returns from benchmark price series
# =========================

benchmark_prices = df_prices_full.iloc[:, 0]

benchmark_oos_records = []

for window_number, (window_start, window_end) in enumerate(rolling_windows, start=1):
    oos_start = window_end + pd.Timedelta(days=1)

    oos_end = (
        oos_start
        + pd.DateOffset(months=rebalancing_period_months)
        - pd.Timedelta(days=1)
    )

    if oos_end > benchmark_prices.index[-1]:
        oos_end = benchmark_prices.index[-1]

    oos_benchmark_prices = benchmark_prices.loc[oos_start:oos_end]

    if oos_benchmark_prices.empty:
        benchmark_return = np.nan
    else:
        benchmark_return = (
            oos_benchmark_prices.iloc[-1] / oos_benchmark_prices.iloc[0]
        ) - 1

    benchmark_oos_records.append(
        {
            "Rolling Window": window_number,
            "benchmark": benchmark_return,
        }
    )

benchmark_oos_returns_df = pd.DataFrame(benchmark_oos_records)


# =========================
# Merge all OOS return blocks into final analysis_df
# =========================

analysis_df = (
    analysis_base_df
    .merge(benchmark_oos_returns_df, on="Rolling Window", how="left")
    .merge(markowitz_oos_returns_df, on="Rolling Window", how="left")
    .merge(risk_parity_oos_returns_df, on="Rolling Window", how="left")
    .merge(cvar_oos_returns_df, on="Rolling Window", how="left")
)

analysis_columns_order = [
    "Rolling Window",
    "Market Volatility",
    "Av.Cr.-Corr.",
    "Market Trend",
    "Skewness",
    "Kurtosis",
    "regime",
    "benchmark",
    "MARK_WF",
    "MARK_WF_RF",
    "MARK15_S",
    "MARK15_S_RF",
    "MARK15_EWMA",
    "MARK5_EWMA",
    "RP_S",
    "RP_EWMA",
    "CVAR 95-10",
    "CVAR 99-15",
]

missing_analysis_columns = [
    column for column in analysis_columns_order
    if column not in analysis_df.columns
]

if missing_analysis_columns:
    raise KeyError(
        "The following columns are missing from analysis_df: "
        + ", ".join(missing_analysis_columns)
    )

analysis_df = analysis_df[analysis_columns_order]


# =========================
# Define return columns for downstream analysis
# =========================

performance_columns = [
    "benchmark",
    "MARK_WF",
    "MARK_WF_RF",
    "MARK15_S",
    "MARK15_S_RF",
    "MARK15_EWMA",
    "MARK5_EWMA",
    "RP_S",
    "RP_EWMA",
    "CVAR 95-10",
    "CVAR 99-15",
]

model_columns_only = [
    "MARK_WF",
    "MARK_WF_RF",
    "MARK15_S",
    "MARK15_S_RF",
    "MARK15_EWMA",
    "MARK5_EWMA",
    "RP_S",
    "RP_EWMA",
    "CVAR 95-10",
    "CVAR 99-15",
]


# =========================
# Final validation
# =========================

if len(analysis_df) != len(rolling_windows):
    raise ValueError(
        "analysis_df row count does not match the number of rolling windows."
    )

if analysis_df[performance_columns].isna().any().any():
    missing_return_columns = analysis_df[performance_columns].columns[
        analysis_df[performance_columns].isna().any()
    ].tolist()

    raise ValueError(
        "Missing values were found in performance return columns: "
        + ", ".join(missing_return_columns)
    )

print("Final analysis_df created successfully.")
print(f"Rows: {analysis_df.shape[0]}")
print(f"Columns: {analysis_df.shape[1]}")

print("\nRegime sequence:")
print(
    analysis_df[["Rolling Window", "regime"]].to_string(index=False)
)

print("\nFinal analysis_df preview:")
print(
    analysis_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)

## Capital Paths of Static Strategies and Benchmark

In [ ]:
# =========================
# Capital Paths of Static Strategies and Benchmark
# =========================

# This cell builds quarterly capital paths for all static strategies
# and the benchmark using the OOS returns stored in analysis_df.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "performance_columns",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Build Final ANALYSIS Dataset section first."
        )


# =========================
# Build quarterly capital paths
# =========================

initial_capital = 100

capital_paths_quarterly_df = pd.DataFrame({
    "Rolling Window": [0] + analysis_df["Rolling Window"].tolist()
})

for column in performance_columns:
    returns = analysis_df[column].astype(float)

    capital_path = [initial_capital]

    for return_value in returns:
        capital_path.append(capital_path[-1] * (1 + return_value))

    capital_paths_quarterly_df[column] = capital_path


# =========================
# Display capital paths
# =========================

print("Quarterly capital paths created successfully.")
print(f"Rows: {capital_paths_quarterly_df.shape[0]}")
print(f"Columns: {capital_paths_quarterly_df.shape[1]}")

print("\nQuarterly capital paths preview:")
print(
    capital_paths_quarterly_df.to_string(
        index=False,
        float_format="{:.6f}".format
    )
)


# =========================
# Plot all capital paths
# =========================

plt.figure(figsize=(18, 10))

for column in performance_columns:
    plt.plot(
        capital_paths_quarterly_df["Rolling Window"],
        capital_paths_quarterly_df[column],
        label=column
    )

plt.title("Quarterly OOS Capital Paths - Static Strategies and Benchmark")
plt.xlabel("Rolling Window")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# =========================
# Optional figure export
# =========================

save_static_capital_paths_plot = False

if save_static_capital_paths_plot:
    try:
        output_path
    except NameError:
        output_path = "output/"

    figure_file = output_path + "STATIC_STRATEGIES_CAPITAL_PATHS.png"

    plt.figure(figsize=(18, 10))

    for column in performance_columns:
        plt.plot(
            capital_paths_quarterly_df["Rolling Window"],
            capital_paths_quarterly_df[column],
            label=column
        )

    plt.title("Quarterly OOS Capital Paths - Static Strategies and Benchmark")
    plt.xlabel("Rolling Window")
    plt.ylabel("Portfolio Value")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(figure_file, dpi=300, bbox_inches="tight")
    plt.show()

    print(f"Figure exported to {figure_file}")

## Table 6 - Overall Performance Metrics

In [ ]:
# =========================
# Table 6 - Overall Performance Metrics
# =========================

# This cell computes overall performance metrics for all static strategies
# and the benchmark using the OOS returns stored in analysis_df.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "performance_columns",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Build Final ANALYSIS Dataset section first."
        )


# =========================
# Helper function
# =========================

def compute_max_drawdown_from_returns(returns):
    """
    Compute max drawdown from a sequence of periodic returns.
    """
    capital_path = (1 + returns).cumprod() * 100

    capital_path = pd.concat(
        [
            pd.Series([100]),
            capital_path.reset_index(drop=True)
        ],
        ignore_index=True
    )

    peak = capital_path.cummax()
    drawdown = (capital_path - peak) / peak

    return drawdown.min()


# =========================
# Compute Table 6 metrics
# =========================

periods_per_year = 4

table_6_records = []

for column in performance_columns:
    returns = analysis_df[column].astype(float).dropna()

    total_return = (1 + returns).prod() - 1
    annual_return = returns.mean() * periods_per_year
    volatility = returns.std(ddof=1) * np.sqrt(periods_per_year)

    if volatility != 0:
        sharpe_like = annual_return / volatility
    else:
        sharpe_like = np.nan

    max_drawdown = compute_max_drawdown_from_returns(returns)

    if max_drawdown != 0:
        calmar = annual_return / abs(max_drawdown)
    else:
        calmar = np.nan

    table_6_records.append(
        {
            "Model": column,
            "Total Return": total_return,
            "Ann.Return": annual_return,
            "Volatility": volatility,
            "Sharpe-like": sharpe_like,
            "MDD": max_drawdown,
            "Calmar": calmar,
        }
    )

table_6_performance_df = pd.DataFrame(table_6_records)

table_6_performance_df = table_6_performance_df.sort_values(
    by="Sharpe-like",
    ascending=False
).reset_index(drop=True)


# =========================
# Display Table 6
# =========================

display(
    table_6_performance_df.style.format(
        {
            "Total Return": "{:.2%}",
            "Ann.Return": "{:.2%}",
            "Volatility": "{:.2%}",
            "Sharpe-like": "{:.2f}",
            "MDD": "{:.2%}",
            "Calmar": "{:.2f}",
        }
    )
)

## Table 7 - Regime-Level Best Model Selection

In [ ]:
# =========================
# Table 7 - Regime-Level Best Model Selection
# =========================

# This cell computes the best model per market regime using:
# - compounded regime-level return,
# - regime-level Sharpe-like score.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "model_columns_only",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Build Final ANALYSIS Dataset section first."
        )


# =========================
# Regime labels
# =========================

regime_name_map = {
    0: "Regime 0 (High Stress)",
    1: "Regime 1 (Low-Risk Growth)",
    2: "Regime 2 (Stable)",
    3: "Regime 3 (Transitional)",
}


# =========================
# Helper function
# =========================

def get_top_model(score_series):
    """
    Return the model with the highest score.
    """
    return score_series.idxmax()


# =========================
# Compute Table 7
# =========================

periods_per_year = 4

table_7_records = []

for regime_value in sorted(analysis_df["regime"].dropna().unique()):
    regime_subset = analysis_df[
        analysis_df["regime"] == regime_value
    ]

    regime_model_returns = regime_subset[model_columns_only].astype(float)

    regime_total_returns = (1 + regime_model_returns).prod() - 1

    regime_annual_returns = regime_model_returns.mean() * periods_per_year
    regime_annual_volatility = regime_model_returns.std(ddof=1) * np.sqrt(periods_per_year)

    regime_sharpe_like = regime_annual_returns / regime_annual_volatility
    regime_sharpe_like = regime_sharpe_like.replace([np.inf, -np.inf], np.nan)

    top_model_return = get_top_model(
        regime_total_returns.dropna()
    )

    top_model_sharpe_like = get_top_model(
        regime_sharpe_like.dropna()
    )

    regime_label = regime_name_map.get(
        int(regime_value),
        f"Regime {int(regime_value)}"
    )

    table_7_records.append(
        {
            "Regime": regime_label,
            "Top Model (Return)": top_model_return,
            "Top Model (Sharpe-like)": top_model_sharpe_like,
        }
    )

table_7_regime_best_model_df = pd.DataFrame(table_7_records)


# =========================
# Display Table 7
# =========================

display(table_7_regime_best_model_df)

## Export Final Analysis Workbook

In [ ]:
# =========================
# Export Final Analysis Workbook
# =========================

# This cell exports the final analysis workbook with:
# - final ANALYSIS dataset,
# - quarterly capital paths,
# - Table 6,
# - Table 7.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "capital_paths_quarterly_df",
    "table_6_performance_df",
    "table_7_regime_best_model_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Fallback setup
# =========================

try:
    output_path
except NameError:
    output_path = "output/"


# =========================
# Export workbook
# =========================

export_final_analysis_workbook = True

if export_final_analysis_workbook:
    export_file = output_path + "ANALYSIS.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        analysis_df.to_excel(
            writer,
            sheet_name="ANALYSIS",
            index=False
        )

        capital_paths_quarterly_df.to_excel(
            writer,
            sheet_name="Capital_Paths",
            index=False
        )

        table_6_performance_df.to_excel(
            writer,
            sheet_name="Table_6_Performance",
            index=False
        )

        table_7_regime_best_model_df.to_excel(
            writer,
            sheet_name="Table_7_Regime_Best",
            index=False
        )

    print(f"Final analysis workbook exported to {export_file}")

# 8. ADAPTIVE PORTFOLIO SIMULATION

## Adaptive Selection Rule

In [ ]:
# =========================
# Adaptive Selection Rule
# =========================

# This cell defines the adaptive portfolio selection rule.
# The selected model for each regime is computed directly from analysis_df.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "model_columns_only",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Selection criterion
# =========================

adaptive_selection_criterion = "return"

if adaptive_selection_criterion not in ["return", "sharpe_like"]:
    raise ValueError(
        "adaptive_selection_criterion must be either 'return' or 'sharpe_like'."
    )


# =========================
# Regime labels
# =========================

regime_name_map = {
    0: "Regime 0 (High Stress)",
    1: "Regime 1 (Low-Risk Growth)",
    2: "Regime 2 (Stable)",
    3: "Regime 3 (Transitional)",
}


# =========================
# Compute best model per regime
# =========================

periods_per_year = 4

best_method_by_regime = {}
adaptive_selection_records = []

for regime_value in sorted(analysis_df["regime"].dropna().unique()):
    regime_subset = analysis_df[
        analysis_df["regime"] == regime_value
    ]

    regime_model_returns = regime_subset[model_columns_only].astype(float)

    regime_total_returns = (1 + regime_model_returns).prod() - 1

    regime_annual_returns = regime_model_returns.mean() * periods_per_year
    regime_annual_volatility = regime_model_returns.std(ddof=1) * np.sqrt(periods_per_year)

    regime_sharpe_like = regime_annual_returns / regime_annual_volatility
    regime_sharpe_like = regime_sharpe_like.replace([np.inf, -np.inf], np.nan)

    return_winner = regime_total_returns.idxmax()
    sharpe_like_winner = regime_sharpe_like.idxmax()

    if adaptive_selection_criterion == "return":
        selected_model = return_winner
    else:
        selected_model = sharpe_like_winner

    best_method_by_regime[int(regime_value)] = selected_model

    adaptive_selection_records.append(
        {
            "Regime": regime_name_map.get(
                int(regime_value),
                f"Regime {int(regime_value)}"
            ),
            "Return Winner": return_winner,
            "Return Winner Value": regime_total_returns[return_winner],
            "Sharpe-like Winner": sharpe_like_winner,
            "Sharpe-like Winner Value": regime_sharpe_like[sharpe_like_winner],
            "Selected Criterion": adaptive_selection_criterion,
            "Selected Model": selected_model,
        }
    )

adaptive_selection_rule_df = pd.DataFrame(adaptive_selection_records)


# =========================
# Display adaptive selection rule
# =========================

display(
    adaptive_selection_rule_df.style.format(
        {
            "Return Winner Value": "{:.2%}",
            "Sharpe-like Winner Value": "{:.2f}",
        }
    )
)

print("Adaptive selection rule:")
print(best_method_by_regime)

## Adaptive Portfolio Returns

In [ ]:
# =========================
# Adaptive Portfolio Returns
# =========================

# This cell assigns the selected model to each rolling window
# and constructs the adaptive portfolio return series.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "best_method_by_regime",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Adaptive Selection Rule section first."
        )


# =========================
# Build adaptive returns
# =========================

adaptive_portfolio_df = analysis_df.copy()

adaptive_portfolio_df["Selected_Method"] = adaptive_portfolio_df["regime"].map(
    best_method_by_regime
)

if adaptive_portfolio_df["Selected_Method"].isna().any():
    raise ValueError(
        "Some regimes do not have a selected model."
    )

adaptive_portfolio_df["Adaptive_Return"] = adaptive_portfolio_df.apply(
    lambda row: row[row["Selected_Method"]],
    axis=1
)

if adaptive_portfolio_df["Adaptive_Return"].isna().any():
    raise ValueError(
        "Adaptive_Return contains missing values."
    )


# =========================
# Build compact adaptive returns table
# =========================

adaptive_returns_df = adaptive_portfolio_df[
    [
        "Rolling Window",
        "regime",
        "Selected_Method",
        "benchmark",
        "Adaptive_Return",
    ]
].copy()


# =========================
# Display adaptive returns
# =========================

display(
    adaptive_returns_df.style.format(
        {
            "benchmark": "{:.2%}",
            "Adaptive_Return": "{:.2%}",
        }
    )
)

## Adaptive Portfolio Capital Path versus Benchmark

In [ ]:
# =========================
# Adaptive Portfolio versus Benchmark
# =========================

# This cell compares the adaptive portfolio against the benchmark.

# =========================
# Required object checks
# =========================

required_objects = [
    "adaptive_portfolio_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Adaptive Portfolio Returns section first."
        )


# =========================
# Helper functions
# =========================

def compute_max_drawdown_from_return_series(returns):
    """
    Compute max drawdown from a periodic return series.
    """
    capital_path = (1 + returns).cumprod()

    peak = capital_path.cummax()
    drawdown = (capital_path - peak) / peak

    return drawdown.min()


def compute_performance_metrics(returns, periods_per_year=4):
    """
    Compute performance metrics from periodic returns.
    """
    returns = returns.dropna().astype(float)

    total_return = (1 + returns).prod() - 1
    annual_return = returns.mean() * periods_per_year
    annual_volatility = returns.std(ddof=1) * np.sqrt(periods_per_year)

    if annual_volatility != 0:
        sharpe_like = annual_return / annual_volatility
    else:
        sharpe_like = np.nan

    max_drawdown = compute_max_drawdown_from_return_series(returns)

    if max_drawdown != 0:
        calmar = annual_return / abs(max_drawdown)
    else:
        calmar = np.nan

    return {
        "Total Return": total_return,
        "Ann.Return": annual_return,
        "Volatility": annual_volatility,
        "Sharpe-like": sharpe_like,
        "MDD": max_drawdown,
        "Calmar": calmar,
    }


# =========================
# Compute adaptive and benchmark cumulative paths
# =========================

adaptive_portfolio_df["Adaptive_Cum"] = (
    1 + adaptive_portfolio_df["Adaptive_Return"]
).cumprod()

adaptive_portfolio_df["Benchmark_Cum"] = (
    1 + adaptive_portfolio_df["benchmark"]
).cumprod()

initial_capital = 100

adaptive_capital_path_df = pd.DataFrame({
    "Rolling Window": [0] + adaptive_portfolio_df["Rolling Window"].tolist(),
    "Adaptive Portfolio": [initial_capital]
    + (adaptive_portfolio_df["Adaptive_Cum"] * initial_capital).tolist(),
    "benchmark": [initial_capital]
    + (adaptive_portfolio_df["Benchmark_Cum"] * initial_capital).tolist(),
})


# =========================
# Compute comparison metrics
# =========================

adaptive_vs_benchmark_records = []

adaptive_metrics = compute_performance_metrics(
    adaptive_portfolio_df["Adaptive_Return"],
    periods_per_year=4
)

adaptive_metrics["Strategy"] = "Adaptive Portfolio"
adaptive_vs_benchmark_records.append(adaptive_metrics)

benchmark_metrics = compute_performance_metrics(
    adaptive_portfolio_df["benchmark"],
    periods_per_year=4
)

benchmark_metrics["Strategy"] = "benchmark"
adaptive_vs_benchmark_records.append(benchmark_metrics)

adaptive_vs_benchmark_metrics_df = pd.DataFrame(
    adaptive_vs_benchmark_records
)

adaptive_vs_benchmark_metrics_df = adaptive_vs_benchmark_metrics_df[
    [
        "Strategy",
        "Total Return",
        "Ann.Return",
        "Volatility",
        "Sharpe-like",
        "MDD",
        "Calmar",
    ]
]


# =========================
# Display comparison metrics
# =========================

display(
    adaptive_vs_benchmark_metrics_df.style.format(
        {
            "Total Return": "{:.2%}",
            "Ann.Return": "{:.2%}",
            "Volatility": "{:.2%}",
            "Sharpe-like": "{:.2f}",
            "MDD": "{:.2%}",
            "Calmar": "{:.2f}",
        }
    )
)


# =========================
# Plot adaptive versus benchmark
# =========================

plt.figure(figsize=(14, 7))

plt.plot(
    adaptive_capital_path_df["Rolling Window"],
    adaptive_capital_path_df["Adaptive Portfolio"],
    label="Adaptive Portfolio",
    linewidth=2.5
)

plt.plot(
    adaptive_capital_path_df["Rolling Window"],
    adaptive_capital_path_df["benchmark"],
    label="benchmark",
    linestyle="--",
    linewidth=2
)

plt.title("Adaptive Portfolio versus Benchmark")
plt.xlabel("Rolling Window")
plt.ylabel("Portfolio Value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Export Adaptive Portfolio Results

In [ ]:
# =========================
# Export Adaptive Portfolio Results
# =========================

# This cell exports the adaptive portfolio simulation results.

# =========================
# Required object checks
# =========================

required_objects = [
    "adaptive_portfolio_df",
    "adaptive_returns_df",
    "adaptive_selection_rule_df",
    "adaptive_capital_path_df",
    "adaptive_vs_benchmark_metrics_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous adaptive portfolio sections first."
        )


# =========================
# Fallback setup
# =========================

try:
    output_path
except NameError:
    output_path = "output/"


# =========================
# Export workbook
# =========================

export_adaptive_portfolio_results = True

if export_adaptive_portfolio_results:
    export_file = output_path + "Adaptive_Portfolio_Simulation.xlsx"

    with pd.ExcelWriter(export_file) as writer:
        adaptive_portfolio_df.to_excel(
            writer,
            sheet_name="Adaptive_Full_Data",
            index=False
        )

        adaptive_returns_df.to_excel(
            writer,
            sheet_name="Adaptive_Returns",
            index=False
        )

        adaptive_selection_rule_df.to_excel(
            writer,
            sheet_name="Selection_Rule",
            index=False
        )

        adaptive_capital_path_df.to_excel(
            writer,
            sheet_name="Capital_Path",
            index=False
        )

        adaptive_vs_benchmark_metrics_df.to_excel(
            writer,
            sheet_name="Adaptive_vs_Benchmark",
            index=False
        )

    print(f"Adaptive portfolio results exported to {export_file}")

# 9. ADAPTIVE PORTFOLIO VERSUS STATIC STRATEGIES

## Consolidated Capital Paths

In [ ]:
# =========================
# Consolidated Capital Paths
# =========================

# This cell combines static strategy capital paths, benchmark,
# and the adaptive portfolio into one consolidated capital path dataframe.

# =========================
# Required object checks
# =========================

required_objects = [
    "capital_paths_quarterly_df",
    "performance_columns",
    "adaptive_portfolio_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Build adaptive capital path if needed
# =========================

if "adaptive_capital_path_df" not in globals():
    adaptive_portfolio_df["Adaptive_Cum"] = (
        1 + adaptive_portfolio_df["Adaptive_Return"]
    ).cumprod()

    adaptive_portfolio_df["Benchmark_Cum"] = (
        1 + adaptive_portfolio_df["benchmark"]
    ).cumprod()

    initial_capital = 100

    adaptive_capital_path_df = pd.DataFrame({
        "Rolling Window": [0] + adaptive_portfolio_df["Rolling Window"].tolist(),
        "Adaptive Portfolio": [initial_capital]
        + (adaptive_portfolio_df["Adaptive_Cum"] * initial_capital).tolist(),
        "benchmark": [initial_capital]
        + (adaptive_portfolio_df["Benchmark_Cum"] * initial_capital).tolist(),
    })


# =========================
# Combine all paths
# =========================

all_capital_paths_df = capital_paths_quarterly_df.copy()

all_capital_paths_df["Adaptive Portfolio"] = adaptive_capital_path_df[
    "Adaptive Portfolio"
].values

all_capital_path_columns = [
    "Adaptive Portfolio",
] + performance_columns


# =========================
# Plot consolidated capital paths
# =========================

plt.figure(figsize=(18, 10))

for column in performance_columns:
    plt.plot(
        all_capital_paths_df["Rolling Window"],
        all_capital_paths_df[column],
        linestyle="--",
        alpha=0.6,
        label=column
    )

plt.plot(
    all_capital_paths_df["Rolling Window"],
    all_capital_paths_df["Adaptive Portfolio"],
    linewidth=3,
    label="Adaptive Portfolio"
)

plt.title("Consolidated Capital Paths - Adaptive Portfolio versus Static Strategies")
plt.xlabel("Rolling Window")
plt.ylabel("Portfolio Value")
plt.legend(ncol=3)
plt.grid(True)
plt.tight_layout()
plt.show()


# =========================
# Display consolidated capital paths
# =========================

display(
    all_capital_paths_df.style.format(
        {
            column: "{:.2f}"
            for column in all_capital_path_columns
            if column in all_capital_paths_df.columns
        }
    )
)

## Performance Comparison Including Adaptive Portfolio

In [ ]:
# =========================
# Performance Comparison Including Adaptive Portfolio
# =========================

# This cell computes performance metrics for all static strategies,
# the benchmark, and the adaptive portfolio.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "performance_columns",
    "adaptive_portfolio_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Helper functions
# =========================

def compute_max_drawdown_from_return_series(returns):
    """
    Compute max drawdown from a periodic return series.
    """
    capital_path = (1 + returns).cumprod()

    peak = capital_path.cummax()
    drawdown = (capital_path - peak) / peak

    return drawdown.min()


def compute_performance_metrics(returns, periods_per_year=4):
    """
    Compute performance metrics from periodic returns.
    """
    returns = returns.dropna().astype(float)

    total_return = (1 + returns).prod() - 1
    annual_return = returns.mean() * periods_per_year
    annual_volatility = returns.std(ddof=1) * np.sqrt(periods_per_year)

    if annual_volatility != 0:
        sharpe_like = annual_return / annual_volatility
    else:
        sharpe_like = np.nan

    max_drawdown = compute_max_drawdown_from_return_series(returns)

    if max_drawdown != 0:
        calmar = annual_return / abs(max_drawdown)
    else:
        calmar = np.nan

    return {
        "Total Return": total_return,
        "Ann.Return": annual_return,
        "Volatility": annual_volatility,
        "Sharpe-like": sharpe_like,
        "MDD": max_drawdown,
        "Calmar": calmar,
    }


# =========================
# Build return series dictionary
# =========================

strategy_return_series = {}

for column in performance_columns:
    strategy_return_series[column] = analysis_df[column].astype(float)

strategy_return_series["Adaptive Portfolio"] = adaptive_portfolio_df[
    "Adaptive_Return"
].astype(float)


# =========================
# Compute performance metrics
# =========================

all_strategy_performance_records = []

for strategy_name, returns in strategy_return_series.items():
    metrics = compute_performance_metrics(
        returns,
        periods_per_year=4
    )

    metrics["Strategy"] = strategy_name
    all_strategy_performance_records.append(metrics)

all_strategy_performance_df = pd.DataFrame(
    all_strategy_performance_records
)

all_strategy_performance_df = all_strategy_performance_df[
    [
        "Strategy",
        "Total Return",
        "Ann.Return",
        "Volatility",
        "Sharpe-like",
        "MDD",
        "Calmar",
    ]
]

all_strategy_performance_df = all_strategy_performance_df.sort_values(
    by="Sharpe-like",
    ascending=False
).reset_index(drop=True)


# =========================
# Display performance comparison
# =========================

display(
    all_strategy_performance_df.style.format(
        {
            "Total Return": "{:.2%}",
            "Ann.Return": "{:.2%}",
            "Volatility": "{:.2%}",
            "Sharpe-like": "{:.2f}",
            "MDD": "{:.2%}",
            "Calmar": "{:.2f}",
        }
    )
)

## Ranking of Strategies

In [ ]:
# =========================
# Ranking of Strategies
# =========================

# This cell ranks all strategies by Sharpe-like performance.

# =========================
# Required object checks
# =========================

required_objects = [
    "all_strategy_performance_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Performance Comparison Including Adaptive Portfolio section first."
        )


# =========================
# Build ranking table
# =========================

strategy_ranking_df = all_strategy_performance_df.copy()

strategy_ranking_df = strategy_ranking_df.sort_values(
    by="Sharpe-like",
    ascending=False
).reset_index(drop=True)

strategy_ranking_df.insert(
    0,
    "Rank",
    range(1, len(strategy_ranking_df) + 1)
)


# =========================
# Display ranking table
# =========================

display(
    strategy_ranking_df.style.format(
        {
            "Total Return": "{:.2%}",
            "Ann.Return": "{:.2%}",
            "Volatility": "{:.2%}",
            "Sharpe-like": "{:.2f}",
            "MDD": "{:.2%}",
            "Calmar": "{:.2f}",
        }
    )
)

# 10. EXTENDED PERFORMANCE ANALYSIS

## DRAWDOWN ANALYSIS


Maximum Drawdown by Strategy

In [ ]:
# =========================
# Maximum Drawdown by Strategy
# =========================

# This cell computes maximum drawdown for all static strategies,
# the benchmark, and the adaptive portfolio.

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "performance_columns",
    "adaptive_portfolio_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Helper function
# =========================

def compute_drawdown_path_from_returns(returns, initial_capital=100):
    """
    Compute capital path and drawdown path from a periodic return series.
    """
    returns = returns.dropna().astype(float)

    capital_path = (1 + returns).cumprod() * initial_capital

    capital_path = pd.concat(
        [
            pd.Series([initial_capital]),
            capital_path.reset_index(drop=True)
        ],
        ignore_index=True
    )

    running_peak = capital_path.cummax()
    drawdown_path = (capital_path - running_peak) / running_peak

    return capital_path, drawdown_path


# =========================
# Build strategy return series
# =========================

drawdown_return_series = {}

for column in performance_columns:
    drawdown_return_series[column] = analysis_df[column].astype(float)

drawdown_return_series["Adaptive Portfolio"] = adaptive_portfolio_df[
    "Adaptive_Return"
].astype(float)


# =========================
# Compute drawdown paths and summary
# =========================

drawdown_paths_df = pd.DataFrame({
    "Rolling Window": [0] + analysis_df["Rolling Window"].tolist()
})

drawdown_summary_records = []

for strategy_name, returns in drawdown_return_series.items():
    capital_path, drawdown_path = compute_drawdown_path_from_returns(
        returns,
        initial_capital=100
    )

    drawdown_paths_df[strategy_name] = drawdown_path.values

    maximum_drawdown = drawdown_path.min()
    maximum_drawdown_window = drawdown_paths_df.loc[
        drawdown_paths_df[strategy_name].idxmin(),
        "Rolling Window"
    ]

    drawdown_summary_records.append(
        {
            "Strategy": strategy_name,
            "Maximum Drawdown": maximum_drawdown,
            "MDD Rolling Window": maximum_drawdown_window,
        }
    )

drawdown_summary_df = pd.DataFrame(drawdown_summary_records)

drawdown_summary_df = drawdown_summary_df.sort_values(
    by="Maximum Drawdown",
    ascending=True
).reset_index(drop=True)


# =========================
# Display maximum drawdown summary
# =========================

display(
    drawdown_summary_df.style.format(
        {
            "Maximum Drawdown": "{:.2%}",
            "MDD Rolling Window": "{:.0f}",
        }
    )
)

Drawdown Path - Adaptive versus Benchmark

In [ ]:
# =========================
# Drawdown Path - Adaptive versus Benchmark
# =========================

# This cell plots the drawdown path of the adaptive portfolio
# against the benchmark.

# =========================
# Required object checks
# =========================

required_objects = [
    "drawdown_paths_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Maximum Drawdown by Strategy section first."
        )


# =========================
# Plot adaptive versus benchmark drawdowns
# =========================

plt.figure(figsize=(14, 7))

plt.plot(
    drawdown_paths_df["Rolling Window"],
    drawdown_paths_df["Adaptive Portfolio"],
    label="Adaptive Portfolio",
    linewidth=2.5
)

plt.plot(
    drawdown_paths_df["Rolling Window"],
    drawdown_paths_df["benchmark"],
    label="benchmark",
    linestyle="--",
    linewidth=2
)

plt.title("Drawdown Path - Adaptive Portfolio versus Benchmark")
plt.xlabel("Rolling Window")
plt.ylabel("Drawdown")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# =========================
# Display selected drawdown paths
# =========================

adaptive_benchmark_drawdown_df = drawdown_paths_df[
    [
        "Rolling Window",
        "Adaptive Portfolio",
        "benchmark",
    ]
].copy()

display(
    adaptive_benchmark_drawdown_df.style.format(
        {
            "Adaptive Portfolio": "{:.2%}",
            "benchmark": "{:.2%}",
        }
    )
)

Drawdown Path - All Strategies

In [ ]:
# =========================
# Drawdown Path - All Strategies
# =========================

# This cell plots drawdown paths for all static strategies,
# the benchmark, and the adaptive portfolio.

# =========================
# Required object checks
# =========================

required_objects = [
    "drawdown_paths_df",
    "performance_columns",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Maximum Drawdown by Strategy section first."
        )


# =========================
# Define plot columns
# =========================

drawdown_plot_columns = [
    "Adaptive Portfolio",
] + performance_columns


# =========================
# Plot all drawdown paths
# =========================

plt.figure(figsize=(18, 10))

for column in performance_columns:
    plt.plot(
        drawdown_paths_df["Rolling Window"],
        drawdown_paths_df[column],
        linestyle="--",
        alpha=0.6,
        label=column
    )

plt.plot(
    drawdown_paths_df["Rolling Window"],
    drawdown_paths_df["Adaptive Portfolio"],
    linewidth=3,
    label="Adaptive Portfolio"
)

plt.title("Drawdown Paths - All Strategies")
plt.xlabel("Rolling Window")
plt.ylabel("Drawdown")
plt.legend(ncol=3)
plt.grid(True)
plt.tight_layout()
plt.show()


# =========================
# Display all drawdown paths
# =========================

display(
    drawdown_paths_df[
        ["Rolling Window"] + drawdown_plot_columns
    ].style.format(
        {
            column: "{:.2%}"
            for column in drawdown_plot_columns
        }
    )
)

## RECOVERY RATIO ANALYSIS

Recovery Ratio by Strategy

In [ ]:
# =========================
# Recovery Ratio by Strategy
# =========================

# This cell computes the recovery ratio for all static strategies,
# the benchmark, and the adaptive portfolio.
# Recovery Ratio = Total Return / Absolute Maximum Drawdown

# =========================
# Required object checks
# =========================

required_objects = [
    "analysis_df",
    "performance_columns",
    "adaptive_portfolio_df",
    "drawdown_summary_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Helper function
# =========================

def compute_total_return_from_returns(returns):
    """
    Compute total compounded return from a periodic return series.
    """
    returns = returns.dropna().astype(float)

    total_return = (1 + returns).prod() - 1

    return total_return


# =========================
# Build strategy return series
# =========================

recovery_return_series = {}

for column in performance_columns:
    recovery_return_series[column] = analysis_df[column].astype(float)

recovery_return_series["Adaptive Portfolio"] = adaptive_portfolio_df[
    "Adaptive_Return"
].astype(float)


# =========================
# Compute recovery ratio
# =========================

recovery_records = []

for strategy_name, returns in recovery_return_series.items():
    total_return = compute_total_return_from_returns(returns)

    maximum_drawdown = drawdown_summary_df.loc[
        drawdown_summary_df["Strategy"] == strategy_name,
        "Maximum Drawdown"
    ].iloc[0]

    if maximum_drawdown == 0:
        recovery_ratio = np.nan
    else:
        recovery_ratio = total_return / abs(maximum_drawdown)

    recovery_records.append(
        {
            "Strategy": strategy_name,
            "Total Return": total_return,
            "Maximum Drawdown": maximum_drawdown,
            "Recovery Ratio": recovery_ratio,
        }
    )

recovery_ratio_df = pd.DataFrame(recovery_records)

recovery_ratio_df = recovery_ratio_df.sort_values(
    by="Recovery Ratio",
    ascending=False
).reset_index(drop=True)


# =========================
# Display recovery ratio table
# =========================

display(
    recovery_ratio_df.style.format(
        {
            "Total Return": "{:.2%}",
            "Maximum Drawdown": "{:.2%}",
            "Recovery Ratio": "{:.2f}",
        }
    )
)

Recovery Ranking

In [ ]:
# =========================
# Recovery Ranking
# =========================

# This cell ranks all strategies by recovery ratio.

# =========================
# Required object checks
# =========================

required_objects = [
    "recovery_ratio_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Recovery Ratio by Strategy section first."
        )


# =========================
# Build recovery ranking
# =========================

recovery_ranking_df = recovery_ratio_df.copy()

recovery_ranking_df = recovery_ranking_df.sort_values(
    by="Recovery Ratio",
    ascending=False
).reset_index(drop=True)

recovery_ranking_df.insert(
    0,
    "Recovery Rank",
    range(1, len(recovery_ranking_df) + 1)
)


# =========================
# Display recovery ranking
# =========================

display(
    recovery_ranking_df.style.format(
        {
            "Total Return": "{:.2%}",
            "Maximum Drawdown": "{:.2%}",
            "Recovery Ratio": "{:.2f}",
        }
    )
)


# =========================
# Plot recovery ranking
# =========================

plt.figure(figsize=(14, 7))

plt.bar(
    recovery_ranking_df["Strategy"],
    recovery_ranking_df["Recovery Ratio"]
)

plt.title("Recovery Ranking by Strategy")
plt.xlabel("Strategy")
plt.ylabel("Recovery Ratio")
plt.xticks(rotation=45, ha="right")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

## RISK-RETURN-RECOVERY MAP

Risk-Return-Recovery Dataset

In [ ]:
# =========================
# Risk-Return-Recovery Dataset
# =========================

# This cell builds the final dataset used for the risk-return-recovery maps.
# It combines return, volatility, drawdown, and recovery ratio metrics
# for all strategies.

# =========================
# Required object checks
# =========================

required_objects = [
    "recovery_ratio_df",
    "all_strategy_performance_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the previous sections first."
        )


# =========================
# Build risk-return-recovery dataset
# =========================

risk_return_recovery_df = all_strategy_performance_df.copy()

risk_return_recovery_df = risk_return_recovery_df.merge(
    recovery_ratio_df[
        [
            "Strategy",
            "Recovery Ratio",
        ]
    ],
    on="Strategy",
    how="left"
)


# =========================
# Validate dataset
# =========================

required_map_columns = [
    "Strategy",
    "Total Return",
    "Volatility",
    "Sharpe-like",
    "MDD",
    "Recovery Ratio",
]

missing_map_columns = [
    column for column in required_map_columns
    if column not in risk_return_recovery_df.columns
]

if missing_map_columns:
    raise KeyError(
        "The following columns are missing from risk_return_recovery_df: "
        + ", ".join(missing_map_columns)
    )

if risk_return_recovery_df[required_map_columns].isna().any().any():
    raise ValueError(
        "Missing values were found in the risk-return-recovery dataset."
    )


# =========================
# Display dataset
# =========================

risk_return_recovery_df = risk_return_recovery_df.sort_values(
    by="Recovery Ratio",
    ascending=False
).reset_index(drop=True)

display(
    risk_return_recovery_df.style.format(
        {
            "Total Return": "{:.2%}",
            "Ann.Return": "{:.2%}",
            "Volatility": "{:.2%}",
            "Sharpe-like": "{:.2f}",
            "MDD": "{:.2%}",
            "Calmar": "{:.2f}",
            "Recovery Ratio": "{:.2f}",
        }
    )
)

Two-Dimensional Risk-Return Map

In [ ]:
# =========================
# Two-Dimensional Risk-Return Map
# =========================

# This cell plots strategies in a two-dimensional risk-return space.
# X-axis: annualized volatility
# Y-axis: annualized return
# Bubble size: recovery ratio

# =========================
# Required object checks
# =========================

required_objects = [
    "risk_return_recovery_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Risk-Return-Recovery Dataset section first."
        )


# =========================
# Prepare plot data
# =========================

plot_df = risk_return_recovery_df.copy()

min_recovery = plot_df["Recovery Ratio"].min()
max_recovery = plot_df["Recovery Ratio"].max()

if max_recovery == min_recovery:
    bubble_sizes = np.repeat(300, len(plot_df))
else:
    bubble_sizes = (
        200
        + 800
        * (
            (plot_df["Recovery Ratio"] - min_recovery)
            / (max_recovery - min_recovery)
        )
    )


# =========================
# Plot two-dimensional risk-return map
# =========================

plt.figure(figsize=(14, 8))

plt.scatter(
    plot_df["Volatility"],
    plot_df["Ann.Return"],
    s=bubble_sizes,
    alpha=0.7,
    edgecolor="k"
)

for _, row in plot_df.iterrows():
    plt.annotate(
        row["Strategy"],
        (
            row["Volatility"],
            row["Ann.Return"],
        ),
        textcoords="offset points",
        xytext=(6, 6),
        ha="left",
        fontsize=9
    )

plt.title("Two-Dimensional Risk-Return Map")
plt.xlabel("Annualized Volatility")
plt.ylabel("Annualized Return")
plt.grid(True, linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

Three-Dimensional Risk-Return-Recovery Map

In [ ]:
# =========================
# Three-Dimensional Risk-Return-Recovery Map
# =========================

# This cell creates a three-dimensional risk-return-recovery map.
# The plot uses:
# X-axis: Volatility
# Y-axis: Total Return
# Z-axis: Recovery Ratio

# =========================
# Required object checks
# =========================

required_objects = [
    "risk_return_recovery_df",
]

for object_name in required_objects:
    if object_name not in globals():
        raise NameError(
            f"Required object '{object_name}' is not defined. "
            "Run the Risk-Return-Recovery Dataset section first."
        )


# =========================
# Prepare plot data
# =========================

df3d = risk_return_recovery_df[
    [
        "Strategy",
        "Total Return",
        "Volatility",
        "Recovery Ratio",
    ]
].copy()

df3d = df3d.sort_values(
    by="Recovery Ratio",
    ascending=False
).reset_index(drop=True)


# =========================
# Create 3D plot
# =========================

fig = plt.figure(figsize=(14, 10))
ax = fig.add_subplot(111, projection="3d")

xs = df3d["Volatility"]
ys = df3d["Total Return"]
zs = df3d["Recovery Ratio"]

colors = plt.cm.tab20(np.linspace(0, 1, len(df3d)))


# =========================
# Plot points and drop lines
# =========================

for i in range(len(df3d)):
    label = df3d.loc[i, "Strategy"]
    x = xs.iloc[i]
    y = ys.iloc[i]
    z = zs.iloc[i]
    color = colors[i]

    ax.scatter(
        x,
        y,
        z,
        color=color,
        marker="o",
        s=200,
        edgecolors="black",
        label=label,
        alpha=0.9
    )

    ax.plot(
        [x, x],
        [y, y],
        [0, z],
        color=color,
        linestyle="--",
        linewidth=2,
        alpha=0.6
    )


# =========================
# Axis formatting
# =========================

ax.set_xlabel("Volatility (Risk)", fontsize=12, labelpad=12)
ax.set_ylabel("Total Return (Reward)", fontsize=12, labelpad=12)
ax.set_zlabel("Recovery Ratio (Resilience)", fontsize=12, labelpad=12)

ax.set_title(
    "3D Map: Risk - Return - Recovery",
    fontsize=16,
    pad=20
)

ax.set_zlim(0, max(df3d["Recovery Ratio"]) * 1.10)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=4,
    fontsize=10,
    title="Strategies"
)

ax.view_init(elev=15, azim=-45)

plt.tight_layout()


# =========================
# Optional export and display figure
# =========================

export_3d_risk_return_recovery_map = False

if export_3d_risk_return_recovery_map:
    filename = output_path + "3D_Risk_Return_Recovery_Map.png"

    plt.savefig(
        filename,
        dpi=300,
        bbox_inches="tight"
    )

    print(f"Figure saved as {filename}")

plt.show()

# 11. FINAL OUTPUTS AND REPRODUCIBILITY CHECKS

## Core Object Validation

In [ ]:
# =========================
# Core Object Validation
# =========================

# This cell validates that the main objects required for thesis replication
# and extended analysis have been created successfully.

required_final_objects = [
    "rolling_windows",
    "analysis_df",
    "table_6_performance_df",
    "table_7_regime_best_model_df",
    "adaptive_portfolio_df",
    "all_strategy_performance_df",
    "strategy_ranking_df",
    "drawdown_summary_df",
    "drawdown_paths_df",
    "recovery_ratio_df",
    "recovery_ranking_df",
    "risk_return_recovery_df",
]

missing_final_objects = [
    object_name
    for object_name in required_final_objects
    if object_name not in globals()
]

if missing_final_objects:
    raise NameError(
        "The following required final objects are missing: "
        + ", ".join(missing_final_objects)
    )

print("All required final objects are available.")
print(f"Number of rolling windows: {len(rolling_windows)}")
print(f"analysis_df rows: {analysis_df.shape[0]}")
print(f"analysis_df columns: {analysis_df.shape[1]}")

if len(rolling_windows) != 22:
    raise ValueError(
        "The number of rolling windows is not equal to 22."
    )

if analysis_df.shape[0] != 22:
    raise ValueError(
        "analysis_df does not contain 22 rows."
    )

print("Core reproducibility checks passed successfully.")

## Reproducibility Summary

In [ ]:
# =========================
# Reproducibility Summary
# =========================

# This cell summarizes the main replicated and extended analysis components.

reproducibility_summary = {
    "Input dataset": data_path,
    "Output folder": output_path,
    "Lookback period months": lookback_period_months,
    "Rebalancing period months": rebalancing_period_months,
    "Number of rebalancing dates": len(rebalancing_dates),
    "Number of rolling windows": len(rolling_windows),
    "Number of strategies in performance comparison": len(all_strategy_performance_df),
    "Table 6 rows": table_6_performance_df.shape[0],
    "Table 7 rows": table_7_regime_best_model_df.shape[0],
    "Drawdown summary rows": drawdown_summary_df.shape[0],
    "Recovery ranking rows": recovery_ranking_df.shape[0],
    "Risk-return-recovery rows": risk_return_recovery_df.shape[0],
}

reproducibility_summary_df = pd.DataFrame(
    list(reproducibility_summary.items()),
    columns=[
        "Item",
        "Value",
    ]
)

display(reproducibility_summary_df)

## Output File Inventory

In [ ]:
# =========================
# Output File Inventory
# =========================

# This cell lists the files currently available in the output folder.

if not os.path.exists(output_path):
    raise FileNotFoundError(
        f"The output folder does not exist: {output_path}"
    )

print("Files available in the output folder:")

output_files = [
    file_name
    for file_name in sorted(os.listdir(output_path))
    if file_name != ".DS_Store"
]

if len(output_files) == 0:
    print("No files found in the output folder.")
else:
    for file_name in output_files:
        print(file_name)